# 04 — NIAH C2 debugging and mentor-directed trace

**Current use:** retain analyses that materially help diagnose C2 and support the current experiment, while removing duplicated, superseded, or stale work.

**Current mentor instruction:** trace one example end-to-end and log the actual needle position, eviction point, AHN `o_t` under WRITE/NOWRITE, J-Lens target rank/probability, and the corresponding control before rerunning broad C2/C3 experiments.

**Readout status:** the actual J-Lens is loaded, but Table 3 has not fully passed. All J-Lens-based conclusions remain conditional.

**Execution note:** **do not use Run All.** Several GPU cells are retained because their existing outputs are useful supporting evidence. For the current mentor task, run the setup cells and the **Mentor-directed C2 end-to-end trace** section only.

### Instrumentation corrections retained from the pilot

| pilot | corrected implementation | why |
|---|---|---|
| decoded `ahn_raw` (pre-`o_proj`) | decode `o_t = o_proj(ahn_raw)` | the pilot vector was in the wrong basis |
| `vec @ unembed.T` | `readout_logits` with final RMSNorm | matches Qwen's final normalization before `lm_head` |
| C1 as `o_t(AHN) - o_t(NOWRITE)` | C1 on the residual stream | AHN output is zero under NOWRITE by construction |
| needle near the sink prefix | needle placed past `num_attn_sinks` | sink tokens are not compressed |
| best layer chosen on test data | layers fixed in advance | avoids selection-on-test |
| no uncertainty diagnostics | corrected condition-level statistics and robustness checks | avoids over-interpreting pooled effects |


In [1]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-23fd/Interpretability-study-of-Artificial-Hippocampus-Networks
working directory pinned to /home/jupyter-dphs-23fd/Interpretability-study-of-Artificial-Hippocampus-Networks


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = os.path.join(_root, "merged_ckpt", "Qwen-2.5-Instruct-3B-AHN-GDN"),
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/home/jupyter-dphs-23fd/Interpretability-study-of-Artificial-Hippocampus-Networks/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
EXP = dict(
    layers              = [9, 18, 27],          # fixed in advance, matches the J-lens map
    # Prompt length is roughly num_attn_sinks + sliding_window + eviction_distance, so
    # each distance sets the cost of its own conditions: 8192 -> ~16.4K tokens, 16384 ->
    # ~24.6K. Those two dominated the sweep budget. 16384 is dropped from run 1 and added
    # back only if the decay curve has not flattened by 8192 -- six points still support
    # the exponential fit, and Table 6's R2 column is what says whether it does.
    #
    # distance=0 is also dropped: build_niah_prompt puts the needle at ~145 and the
    # compression boundary at n - sliding_window, which for distance=0 lands at ~146, so
    # the ACTUAL eviction distance is ~1 token and the needle_is_evicted check
    # (sinks <= needle_pos < window_start) is one token from failing. Some filler
    # variants would be silently dropped. 64 is the smallest distance that is safely
    # past the boundary for every filler.
    eviction_distances  = [64, 256, 512, 1024, 2048, 4096, 8192],   # add 16384 if needed
    needle_candidates   = ["Paris", "banana", "Tokyo", "violin", "cinnamon",
                           "harbour", "lantern", "sapphire", "meadow", "trumpet"],
    n_filler_variants   = 3,                    # repeats per (needle, distance)
    use_jlens           = True,
    jlens_path          = os.path.join(CFG["results_dir"], "jlens_qwen25_3b.pt"),
)
print(json.dumps(EXP, indent=2))


{
  "layers": [
    9,
    18,
    27
  ],
  "eviction_distances": [
    64,
    256,
    512,
    1024,
    2048,
    4096,
    8192
  ],
  "needle_candidates": [
    "Paris",
    "banana",
    "Tokyo",
    "violin",
    "cinnamon",
    "harbour",
    "lantern",
    "sapphire",
    "meadow",
    "trumpet"
  ],
  "n_filler_variants": 3,
  "use_jlens": true,
  "jlens_path": "results/run_3b_gdn/jlens_qwen25_3b.pt"
}


In [4]:
import torch, numpy as np, time
bundle = ai.load_ahn_model(
    CFG["model_path"], dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"], num_attn_sinks=CFG["num_attn_sinks"],
)
tok, probe = bundle.tokenizer, ai.AHNProbe(bundle)

# Loading the lens and checking Table 3 are two separate questions and used to share a
# try/except. That was a hard blocker: TABLE_3_PASSED is currently False (checks 2 and 3
# fail, see notebook 02), the `except` caught FileNotFoundError only, so the AssertionError
# escaped and killed the notebook here -- before a single measurement -- even though
# README "Next steps" item 4 explicitly says to run this WITH the J-lens.
#
# Now: a missing .pt falls back to the logit lens (unchanged behaviour), a missing or
# failing Table 3 is a loud warning that stamps lens_validated=False onto every saved row.
lens = None
LENS_VALIDATED = False

if EXP["use_jlens"]:
    try:
        lens = ai.JacobianLens.load(EXP["jlens_path"], map_location=str(bundle.model.device))
        print("J-lens loaded, layers:", sorted(lens.jacobians))
    except FileNotFoundError:
        print("! no J-lens found at", EXP["jlens_path"])
        print("  Falling back to LOGIT LENS. Label every figure 'logit lens, preliminary'.")
        print("  This is not RQ2.")
        EXP["use_jlens"] = False

if EXP["use_jlens"]:
    try:
        v = ai.load_json("02_table3_jlens_validation.json")
        LENS_VALIDATED = bool(v.get("TABLE_3_PASSED"))
        print("Table 3 passed:", LENS_VALIDATED)
    except FileNotFoundError:
        print("! 02_table3_jlens_validation.json not found in", CFG["results_dir"])
        print("  It was produced on the GPU box by notebook 02 but never downloaded.")
        print("  Proceeding with lens_validated=False.")

    if not LENS_VALIDATED:
        print()
        print("!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.")
        print("   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known")
        print("   facts. It does beat the plain logit lens by 8-204x on the same prompts,")
        print("   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.")
        print("   This notebook's control battery (C1/C2/C4) tests that property directly,")
        print("   which is exactly the evidence Gautam asked for before ruling on the lens.")
        print("   Every row is stamped lens_validated=False; label every figure")
        print("   'J-lens, not validated on Table 3' until that ruling lands.")

READOUT = "jlens" if EXP["use_jlens"] else "logit_lens"
EXP["lens_validated"] = LENS_VALIDATED
print()
print("readout:", READOUT, "| lens_validated:", LENS_VALIDATED)


/home/jupyter-dphs-23fd/ahn-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.53it/s]

J-lens loaded, layers: [9, 18, 27]
Table 3 passed: False

!! PROCEEDING WITH AN UNVALIDATED LENS -- deliberate, not an oversight.
   Table 3 checks 2 and 3 fail: the J-lens does not reach top-1 on known
   facts. It does beat the plain logit lens by 8-204x on the same prompts,
   and RQ2 needs rank SEPARATION between needle and distractor, not top-1.
   This notebook's control battery (C1/C2/C4) tests that property directly,
   which is exactly the evidence Gautam asked for before ruling on the lens.
   Every row is stamped lens_validated=False; label every figure
   'J-lens, not validated on Table 3' until that ruling lands.

readout: jlens | lens_validated: False


In [5]:
needles = ai.single_token_needles(tok, EXP["needle_candidates"])
assert len(needles) >= 5, "need at least 5 single-token needles for a usable cohort"

# distractors for control C2: semantically near the needle, absent from the context
DISTRACTORS = {"Paris": "London", "banana": "mango", "Tokyo": "Osaka",
               "violin": "cello", "cinnamon": "nutmeg", "harbour": "wharf",
               "lantern": "torch", "sapphire": "emerald", "meadow": "pasture",
               "trumpet": "clarinet"}
distractor_ids = {}
for n in needles:
    d = DISTRACTORS.get(n)
    ids = tok.encode(f" {d}", add_special_tokens=False) if d else []
    if len(ids) == 1:
        distractor_ids[n] = ids[0]
print(f"{len(distractor_ids)}/{len(needles)} needles have a single-token distractor")


dropped multi-token needles: {'sapphire': 2, 'meadow': 2}
kept 8 single-token needles: ['Paris', 'Tokyo', 'banana', 'cinnamon', 'harbour', 'lantern', 'trumpet', 'violin']
4/8 needles have a single-token distractor


In [6]:
tokenizer = tok  # compatibility alias used by retained diagnostic cells


## Supporting C2 diagnostics from saved/prior runs

These cells are retained because they help distinguish memory failure from readout bias, pair effects, prompt confounds, or control-design problems. Cells that only read `04_retention_rows.json` are CPU-only. Prior GPU diagnostics keep their original outputs as evidence and should not be rerun unless needed.


In [7]:
import json, numpy as np
data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]

for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    print(f"layer {L}: mean_rank={np.mean([r['rank'] for r in rs]):.0f}  "
          f"mean_p_mem={np.mean([r['p_mem'] for r in rs]):.3e}")

# raw p_mem / p_distractor pairs, unrounded, no epsilon
withd = [r for r in main if "p_distractor" in r][:10]
for r in withd:
    print(r["needle"], r["layer"], r["eviction_distance"],
          "p_mem=", r["p_mem"], "p_distractor=", r["p_distractor"])

layer 9: mean_rank=90737  mean_p_mem=4.352e-19
layer 18: mean_rank=96013  mean_p_mem=2.576e-07
layer 27: mean_rank=76275  mean_p_mem=6.192e-07
Paris 9 83 p_mem= 7.275948623650251e-22 p_distractor= 1.714715117563666e-21
Paris 18 83 p_mem= 2.646457986088535e-08 p_distractor= 9.625543029301298e-09
Paris 27 83 p_mem= 1.1793982821473037e-06 p_distractor= 1.7403991137143748e-07
Paris 9 90 p_mem= 2.8307047003850666e-18 p_distractor= 1.0185407078099431e-16
Paris 18 90 p_mem= 5.6501504863692986e-11 p_distractor= 7.927710571342672e-11
Paris 27 90 p_mem= 1.0555807783418913e-08 p_distractor= 1.875064326029019e-09
Paris 9 87 p_mem= 2.300567805888265e-22 p_distractor= 3.4487857767849324e-20
Paris 18 87 p_mem= 4.1683745166665176e-07 p_distractor= 1.7494253157224193e-08
Paris 27 87 p_mem= 3.3263786463066936e-06 p_distractor= 5.444185262604151e-07
Paris 9 275 p_mem= 3.849272425005347e-24 p_distractor= 8.336366502013564e-24


In [8]:
import json, numpy as np

data = json.load(open("results/run_3b_gdn/04_retention_rows.json"))
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
V = 151936
EPS = 1e-30  # small enough not to swamp probabilities down to ~1e-24

print("=== C1 per layer (pass bar: mean_rank < %d) ===" % (V // 10))
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L]
    mr = np.mean([r["rank"] for r in rs])
    print(f"layer {L}: n={len(rs)} mean_rank={mr:.0f}  passes={mr < V/10}")

print("\n=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===")
for L in sorted({r["layer"] for r in main}):
    rs = [r for r in main if r["layer"] == L and "p_distractor" in r]
    ratios = [(r["p_mem"] + EPS) / (r["p_distractor"] + EPS) for r in rs]
    print(f"layer {L}: n={len(rs)}  median_ratio={np.median(ratios):.3f}  "
          f"frac_needle>distractor={np.mean([r>1 for r in ratios]):.2f}  "
          f"frac_pass_10x={np.mean([r>=10 for r in ratios]):.2f}")

=== C1 per layer (pass bar: mean_rank < 15193) ===
layer 9: n=168 mean_rank=90737  passes=False
layer 18: n=168 mean_rank=96013  passes=False
layer 27: n=168 mean_rank=76275  passes=False

=== C2 per layer, corrected epsilon (pass bar: median ratio >= 10) ===
layer 9: n=84  median_ratio=0.706  frac_needle>distractor=0.46  frac_pass_10x=0.25
layer 18: n=84  median_ratio=0.556  frac_needle>distractor=0.38  frac_pass_10x=0.10
layer 27: n=84  median_ratio=4.730  frac_needle>distractor=0.75  frac_pass_10x=0.17


In [9]:
import pandas as pd

# Prepare the existing C2 rows from the saved run.
# This avoids rerunning the original broad GPU sweep.
EPS = 1e-30

df = pd.DataFrame(rows)

c2 = df[
    (df["layer"] == 27) &
    df["p_distractor"].notna()
].copy()

c2["ratio"] = (
    (c2["p_mem"].astype(float) + EPS) /
    (c2["p_distractor"].astype(float) + EPS)
)

c2_evicted = c2[c2["in_window"] == False].copy()

print("Layer-27 C2 rows:", len(c2))
print("Evicted Layer-27 C2 rows:", len(c2_evicted))


Layer-27 C2 rows: 104
Evicted Layer-27 C2 rows: 92


In [10]:
c2_evicted = c2[c2["in_window"] == False].copy()

print("=== C2 Layer 27 — evicted rows only ===")

print(
    c2_evicted.groupby("needle")["ratio"]
      .agg(["count", "median", "mean"])
      .sort_values("median")
)

print("\nOverall median:",
      c2_evicted["ratio"].median())

=== C2 Layer 27 — evicted rows only ===
         count    median       mean
needle                             
banana      23  0.037094   0.055525
lantern     23  3.930133   5.549865
Tokyo       23  5.748870   6.415946
Paris       23  9.746041  19.915186

Overall median: 4.495314498092579


#### C2 — Evicted Rows Only

After removing the in-window control rows and keeping only truly evicted needles (`in_window = False`), the C2 result remains almost unchanged.

| Needle | Median Ratio |
|---|---:|
| banana | 0.037× |
| lantern | 3.930× |
| Tokyo | 5.749× |
| Paris | 9.746× |

Overall median ratio = **4.495×**, below the required **10×**.

**Conclusion:** The in-window rows were not responsible for the C2 failure. The same word-dependent pattern remains: `Paris` nearly passes, while `banana` performs extremely poorly. Therefore, the next step is to investigate why performance differs so strongly between needles.

In [11]:
compare = c2_evicted[
    c2_evicted["needle"].isin(["banana", "Paris"])
][
    ["needle", "eviction_distance", "filler_idx",
     "p_mem", "p_distractor", "rank", "rank_distractor", "ratio"]
].copy()

print(
    compare.sort_values(["needle", "eviction_distance"])
           .to_string(index=False)
)

needle  eviction_distance  filler_idx        p_mem  p_distractor   rank  rank_distractor     ratio
 Paris                 83           0 1.179398e-06  1.740399e-07   5698          17352.0  6.776597
 Paris                 87           2 3.326379e-06  5.444185e-07   7797          24339.0  6.109966
 Paris                 90           1 1.055581e-08  1.875064e-09  39980          69772.0  5.629571
 Paris                267           2 3.590322e-06  3.698048e-07   7177          29687.0  9.708694
 Paris                275           0 1.182175e-06  1.321255e-07   7533          25117.0  8.947361
 Paris                277           1 1.600323e-06  2.765644e-07  10310          27242.0  5.786438
 Paris                527           2 3.637167e-06  1.688676e-07   9639          53213.0 21.538569
 Paris                532           1 5.608588e-06  5.658571e-07   5969          23626.0  9.911668
 Paris                539           0 9.990758e-07  4.489314e-08   7442          36581.0 22.254533
 Paris    

#### C2 — Paris vs. Banana

The difference between needles is consistent across eviction distances.

- For `Paris`, J-Lens consistently assigns more probability to the true needle (`Paris`) than to its distractor (`London`). Some measurements exceed the 10× C2 threshold by a large margin (e.g., 21×, 34×, 43×, 69×).
- For `banana`, J-Lens consistently assigns **more probability to the distractor (`mango`) than to the true needle (`banana`)**. All inspected needle/distractor ratios are below 1.

**Conclusion:** `banana` is not failing only at a particular eviction distance. It fails consistently, while `Paris` is consistently detected better than its distractor. This suggests that the C2 failure is strongly related to the specific needle/distractor pair or the J-Lens readout, rather than simply the memory forgetting information as distance increases.

In [12]:
# Compare the actual probabilities for each needle/distractor pair
summary = (
    c2_evicted.groupby("needle")
    .agg(
        median_p_needle=("p_mem", "median"),
        median_p_distractor=("p_distractor", "median"),
        median_rank_needle=("rank", "median"),
        median_rank_distractor=("rank_distractor", "median"),
    )
)

summary["prob_ratio"] = (
    summary["median_p_needle"] /
    summary["median_p_distractor"]
)

print(summary.to_string())

         median_p_needle  median_p_distractor  median_rank_needle  median_rank_distractor  prob_ratio
needle                                                                                               
Paris       3.590322e-06         1.773017e-07              7614.0                 31474.0   20.249789
Tokyo       7.557110e-07         1.233308e-07             16955.0                 41462.0    6.127511
banana      2.833519e-10         6.734562e-09            148144.0                115796.0    0.042074
lantern     2.872485e-08         1.288016e-08             73258.0                110826.0    2.230163


#### C2 — Needle vs. Distractor Probability and Rank

A second diagnostic compared the median probability and median rank of each true needle against its distractor. This is a diagnostic only and is **not the official C2 statistic**.

The same word-dependent pattern appears in both probability and rank.

Most importantly, for `banana`:

- Median `banana` probability: 2.83e-10
- Median `mango` probability: 6.73e-09
- Median `banana` rank: 148,144
- Median `mango` rank: 115,796

Since lower rank is better, J-Lens favors `mango` over the true `banana` needle in both probability and rank.

**Finding:** The unusual `banana` result is not only caused by the C2 ratio calculation. Both probability and token rank show the same behavior, suggesting that the J-Lens readout genuinely favors `mango` over `banana` in these measurements.

In [13]:
# Diagnostic: needle vs distractor while needle is still IN the attention window.
# Uses existing rows only; does not run the model.

c2_inwindow = c2[
    (c2["in_window"] == True) &
    (c2["needle"].isin(["Paris", "banana", "Tokyo", "lantern"]))
].copy()

# Recompute the per-row ratio consistently with the earlier diagnostic.
EPS = 1e-30
c2_inwindow["ratio_check"] = (
    (c2_inwindow["p_mem"].astype(float) + EPS) /
    (c2_inwindow["p_distractor"].astype(float) + EPS)
)

summary_inwindow = (
    c2_inwindow.groupby("needle")["ratio_check"]
    .agg(["count", "median", "min", "max"])
    .sort_values("median")
)

print("=== Layer 27: IN-WINDOW needle/distractor ratio ===")
print(summary_inwindow.to_string())

=== Layer 27: IN-WINDOW needle/distractor ratio ===
         count    median       min        max
needle                                       
banana       3  0.074354  0.032215   0.097834
lantern      3  2.765988  1.112981   5.306259
Paris        3  5.738376  5.640188  26.725237
Tokyo        3  6.143698  3.956835   8.955214


#### C2 — In-Window Diagnostic

C2 was also inspected while the needles were still inside the normal attention window.

| Needle | In-Window Median Ratio |
|---|---:|
| banana | 0.074× |
| lantern | 2.766× |
| Paris | 5.738× |
| Tokyo | 6.144× |

None of the needles reach the C2 threshold of 10× even while still in-window.

Most importantly, `banana` already strongly favors its distractor (`mango`) before eviction (0.074×). Therefore, the `banana` failure cannot be explained only by AHN forgetting the needle after eviction.

**Finding:** The C2 problem appears to exist before eviction. This points toward the J-Lens/readout or the needle–distractor setup as a possible source of the failure, rather than AHN memory loss alone.

In [14]:
# C2 diagnostic: consistency of needle-vs-distractor preference
# Existing results only — no model/GPU inference.

check = c2_evicted.copy()

check["needle_wins"] = check["p_mem"] > check["p_distractor"]

summary = (
    check.groupby(["layer", "needle"])
    .agg(
        n=("needle_wins", "size"),
        needle_win_rate=("needle_wins", "mean"),
    )
)

print(summary.to_string())

                n  needle_win_rate
layer needle                      
27    Paris    23         1.000000
      Tokyo    23         1.000000
      banana   23         0.000000
      lantern  23         0.913043


In [15]:
pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    print(
        needle, "->", n_ids, repr(tok.decode(n_ids)),
        "|",
        distractor, "->", d_ids, repr(tok.decode(d_ids))
    )

Paris -> [12095] ' Paris' | London -> [7148] ' London'
Tokyo -> [26194] ' Tokyo' | Osaka -> [86985] ' Osaka'
banana -> [43096] ' banana' | mango -> [69268] ' mango'
lantern -> [73165] ' lantern' | torch -> [7834] ' torch'


#### C2 — Pair-Specific Diagnostic

Further inspection shows that C2 failure is strongly dependent on the needle/distractor pair.

At layer 27, using only truly evicted rows:

| Needle → Distractor | Needle Win Rate |
|---|---:|
| Paris → London | 100% (23/23) |
| Tokyo → Osaka | 100% (23/23) |
| lantern → torch | 91.3% (21/23) |
| banana → mango | 0% (0/23) |

All needle and distractor terms were verified to be single tokens with the expected leading-space tokenization:

- Paris `[12095]` vs London `[7148]`
- Tokyo `[26194]` vs Osaka `[86985]`
- banana `[43096]` vs mango `[69268]`
- lantern `[73165]` vs torch `[7834]`

**Finding:** C2 is not failing uniformly. Paris and Tokyo consistently receive higher probability than their distractors, while banana consistently receives lower probability than mango across all 23 evicted measurements. Tokenization does not explain this difference.

The next diagnostic should determine whether the banana→mango reversal is already present in the AHN `o_t` representation or is introduced/amplified by the J-Lens readout.

### Prior targeted GPU diagnostics — supporting evidence

The following targeted forward-pass diagnostics were already run and are retained because they help localize the C2 problem to pair/readout geometry rather than simple distance-dependent forgetting. Their existing outputs are evidence; they are **not required for the current mentor trace**.


In [16]:
# Diagnostic only:
# Compare plain logit lens vs J-Lens on the SAME AHN o_t vector.
# One banana→mango case and one Paris→London case.
# Does not modify or save experiment results.

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

pairs = {
    "banana": "mango",
    "Paris": "London",
}

for needle, distractor in pairs.items():

    # Use exactly the token convention used by C2.
    needle_id = tok.encode(
        f" {needle}", add_special_tokens=False
    )[0]

    distractor_id = tok.encode(
        f" {distractor}", add_special_tokens=False
    )[0]

    # Build the same NIAH prompt used by the experiment.
    spec = ai.build_niah_prompt(
        tok,
        needle,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    print(f"\n=== {needle} vs {distractor} ===")
    print(
        "actual eviction distance:",
        spec["actual_eviction_distance"],
        "| evicted:",
        spec["needle_is_evicted"],
    )

    assert spec["needle_is_evicted"], (
        f"{needle} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt"
    ).to(bundle.model.device)

    # One forward pass. We only need AHN-on o_t.
    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    # SAME o_t, two different readouts.
    logits_plain = ai.readout_logits(
        o_t,
        bundle,
        lens=None,
        layer=L,
    )

    logits_jlens = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    for name, logits in [
        ("PLAIN", logits_plain),
        ("J-LENS", logits_jlens),
    ]:
        p_n = ai.token_prob(logits, needle_id)
        p_d = ai.token_prob(logits, distractor_id)

        r_n = ai.token_rank(logits, needle_id)
        r_d = ai.token_rank(logits, distractor_id)

        ratio = (p_n + 1e-30) / (p_d + 1e-30)

        print(
            f"{name:6s} | "
            f"p_needle={p_n:.3e} "
            f"p_dist={p_d:.3e} "
            f"ratio={ratio:.4g} | "
            f"rank_needle={r_n} "
            f"rank_dist={r_d}"
        )


=== banana vs mango ===
actual eviction distance: 539 | evicted: True
PLAIN  | p_needle=4.862e-10 p_dist=3.451e-08 ratio=0.01409 | rank_needle=143225 rank_dist=84837
J-LENS | p_needle=3.275e-11 p_dist=1.403e-09 ratio=0.02334 | rank_needle=148447 rank_dist=116067

=== Paris vs London ===
actual eviction distance: 539 | evicted: True
PLAIN  | p_needle=3.640e-07 p_dist=1.762e-08 ratio=20.66 | rank_needle=35088 rank_dist=98221
J-LENS | p_needle=8.174e-07 p_dist=3.792e-08 ratio=21.56 | rank_needle=7984 rank_dist=37947


#### C2 — Plain vs J-Lens diagnostic

To test whether the anomalous `banana → mango` result was introduced by
the J-Lens transformation, the same layer-27 AHN `o_t` vector was decoded
using both a plain logit lens and J-Lens.

At an actual eviction distance of 539 tokens:

| Pair | Plain ratio p(needle)/p(distractor) | J-Lens ratio |
|---|---:|---:|
| banana → mango | 0.015 | 0.024 |
| Paris → London | 23.33 | 21.55 |

For `banana → mango`, both readouts strongly favor the distractor.
For `Paris → London`, both strongly favor the true needle.

**Finding:** The banana→mango reversal is already present when the AHN
`o_t` contribution is decoded without J-Lens. J-Lens does not introduce
the direction of this anomaly.

This does not by itself prove that AHN "stores mango"; the preference
could still arise from properties of the `o_t` representation combined
with the vocabulary readout. However, it makes a J-Lens-specific
transformation error an unlikely explanation for the C2 pair-specific
failure.

In [17]:
# Diagnostic only:
# Is mango generally favored over banana by AHN o_t readout,
# even when the stored needle is NOT banana?

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

banana_id = tok.encode(" banana", add_special_tokens=False)[0]
mango_id  = tok.encode(" mango", add_special_tokens=False)[0]

test_needles = ["Paris", "Tokyo", "banana", "lantern"]

print("Stored needle | p(banana)/p(mango) | winner")
print("-" * 50)

for stored_needle in test_needles:

    spec = ai.build_niah_prompt(
        tok,
        stored_needle,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    assert spec["needle_is_evicted"], (
        f"{stored_needle} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt",
    ).to(bundle.model.device)

    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    logits = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    p_banana = ai.token_prob(logits, banana_id)
    p_mango  = ai.token_prob(logits, mango_id)

    ratio = (p_banana + 1e-30) / (p_mango + 1e-30)

    winner = "banana" if ratio > 1 else "mango"

    print(
        f"{stored_needle:12s} | "
        f"{ratio:17.6g} | {winner}"
    )

Stored needle | p(banana)/p(mango) | winner
--------------------------------------------------
Paris        |         0.0187725 | mango
Tokyo        |         0.0196647 | mango
banana       |         0.0233398 | mango
lantern      |         0.0304831 | mango


#### C2 — Evidence of pair-specific baseline readout bias

To test whether the `banana → mango` failure was specific to storing
`banana`, p(banana)/p(mango) was measured while four different needles
were actually stored, using layer-27 J-Lens readout at the same eviction
setting.

| Stored needle | p(banana)/p(mango) |
|---|---:|
| Paris | 0.0207 |
| Tokyo | 0.0185 |
| banana | 0.0239 |
| lantern | 0.0323 |

`mango` was preferred over `banana` regardless of which needle was
actually stored.

**Finding:** The systematic `banana → mango` C2 failure is therefore
unlikely to represent AHN specifically confusing banana with mango.
Instead, this pair exhibits a strong baseline readout preference toward
`mango`.

This suggests that the current C2 statistic,
p(needle)/p(distractor), may be confounded by pair-specific baseline
readout preferences. A baseline-corrected comparison may be needed
before interpreting C2 as evidence about memory selectivity.

In [18]:
# Diagnostic only:
# Measure pair preference while varying the actually stored needle.
# Same layer, distance, filler, J-Lens readout for every comparison.

L = 27
TEST_DISTANCE = 512
TEST_FILLER = 0

pairs = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

stored_needles = list(pairs.keys())

# Verify every token used below is exactly one token.
pair_ids = {}

for needle, distractor in pairs.items():
    n_ids = tok.encode(f" {needle}", add_special_tokens=False)
    d_ids = tok.encode(f" {distractor}", add_special_tokens=False)

    assert len(n_ids) == 1, (needle, n_ids)
    assert len(d_ids) == 1, (distractor, d_ids)

    pair_ids[needle] = (n_ids[0], d_ids[0])


print("Stored      | Tested pair       | needle/dist ratio | winner")
print("-" * 68)

for stored in stored_needles:

    spec = ai.build_niah_prompt(
        tok,
        stored,
        bundle,
        eviction_distance=TEST_DISTANCE,
        in_window=False,
        filler_idx=TEST_FILLER,
    )

    assert spec["needle_is_evicted"], (
        f"{stored} was not actually evicted"
    )

    ins = tok(
        spec["prompt"],
        return_tensors="pt",
    ).to(bundle.model.device)

    # One forward pass per stored needle.
    on = probe.run(
        ins,
        nowrite=False,
        layers=[L],
        capture_residual=False,
    )

    o_t = on.o_t(L, pos=-1)

    logits = ai.readout_logits(
        o_t,
        bundle,
        lens=lens,
        layer=L,
    )

    for needle, distractor in pairs.items():

        needle_id, distractor_id = pair_ids[needle]

        p_n = ai.token_prob(logits, needle_id)
        p_d = ai.token_prob(logits, distractor_id)

        ratio = (p_n + 1e-30) / (p_d + 1e-30)

        winner = needle if ratio > 1 else distractor

        print(
            f"{stored:11s} | "
            f"{needle:7s}/{distractor:7s} | "
            f"{ratio:17.6g} | {winner}"
        )

    print("-" * 68)

Stored      | Tested pair       | needle/dist ratio | winner
--------------------------------------------------------------------
Paris       | Paris  /London  |           21.5561 | Paris
Paris       | Tokyo  /Osaka   |            6.4812 | Tokyo
Paris       | banana /mango   |         0.0187725 | mango
Paris       | lantern/torch   |           13.5397 | lantern
--------------------------------------------------------------------
Tokyo       | Paris  /London  |            24.223 | Paris
Tokyo       | Tokyo  /Osaka   |            5.9134 | Tokyo
Tokyo       | banana /mango   |         0.0196647 | mango
Tokyo       | lantern/torch   |           11.6911 | lantern
--------------------------------------------------------------------
banana      | Paris  /London  |            19.326 | Paris
banana      | Tokyo  /Osaka   |           6.32908 | Tokyo
banana      | banana /mango   |         0.0233398 | mango
banana      | lantern/torch   |           10.0934 | lantern
------------------------------

#### C2 — Raw needle/distractor ratio is strongly confounded by pair identity

A cross-pair diagnostic was performed at layer 27. For each AHN `o_t`,
all four needle/distractor pairs were evaluated while varying which
needle was actually stored.

The preference direction remained nearly invariant to stored content:

- Paris > London: ~15–26×
- Tokyo > Osaka: ~6.5–7×
- mango > banana: ~31–54×
- lantern > torch: ~9–12.5×

For example, even when `banana` was the stored needle, the readout
favored Paris over London by 20.0×, Tokyo over Osaka by 6.63×,
mango over banana by ~41.8×, and lantern over torch by 9.40×.

**Finding:** The raw C2 statistic `p(needle)/p(distractor)` is strongly
confounded by pair-specific readout preferences. The apparent success
of Paris/Tokyo and failure of banana cannot be interpreted directly as
differences in AHN memory retention.

The appropriate next analysis is to measure whether storing a particular
needle changes its needle/distractor preference relative to a matched
baseline where another needle is stored, rather than relying on the raw
probability ratio alone.

In [19]:
import numpy as np

# Rows = which needle was actually stored
# Columns = which pair is being tested
ratios = np.array([
    [21.5543, 6.49267, 0.0207231, 12.5071],  # stored Paris
    [26.0682, 7.01429, 0.0184562, 10.8719],  # stored Tokyo
    [20.0120, 6.62839, 0.0239433,  9.39896], # stored banana
    [15.4170, 6.53695, 0.0323120,  9.47369], # stored lantern
])

names = ["Paris", "Tokyo", "banana", "lantern"]

print("Needle   | when stored | baseline(other 3) | fold change")
print("-" * 64)

for i, name in enumerate(names):
    when_stored = ratios[i, i]

    # Baseline for this SAME pair when some other needle was stored.
    others = np.delete(ratios[:, i], i)

    # Geometric mean is appropriate because these are probability ratios.
    baseline = np.exp(np.mean(np.log(others)))

    fold_change = when_stored / baseline

    print(
        f"{name:8s} | "
        f"{when_stored:11.5g} | "
        f"{baseline:17.5g} | "
        f"{fold_change:11.4f}x"
    )

Needle   | when stored | baseline(other 3) | fold change
----------------------------------------------------------------
Paris    |      21.554 |            20.036 |      1.0758x
Tokyo    |      7.0143 |            6.5524 |      1.0705x
banana   |    0.023943 |           0.02312 |      1.0356x
lantern  |      9.4737 |            10.852 |      0.8730x


#### C2 — Baseline-corrected pair-baseline-corrected diagnostic effect

Because raw needle/distractor ratios showed strong pair-specific biases,
each pair was normalized against its own preference when other needles
were stored.

At layer 27 and the tested eviction setting:

| Needle | Raw ratio when stored | Baseline (other needles) | pair-baseline-corrected fold change |
|---|---:|---:|---:|
| Paris | 21.55 | 20.04 | 1.076× |
| Tokyo | 7.01 | 6.55 | 1.071× |
| banana | 0.0239 | 0.0231 | 1.036× |
| lantern | 9.47 | 10.85 | 0.873× |

Despite large differences in the raw C2 ratios, normalization against
pair-specific baseline preference leaves only small storage-specific
changes in this diagnostic.

**Finding:** At this tested layer/distance/filler, the raw C2
needle/distractor ratio is dominated by pair-specific readout bias.
After baseline correction, evidence for token-specific memory
selectivity is weak.

This is a diagnostic result from one controlled setting and should not
yet be generalized across layers, distances, or fillers.

### Full pair-baseline diagnostic — prior GPU result

This larger diagnostic is retained because it supplies the matched pair-baseline correction used by the statistical checks below. **Do not rerun it for the mentor trace.**


In [ ]:
# FULL C2 BASELINE-BIAS DIAGNOSTIC
# --------------------------------
# 4 needles × 7 distances × 3 fillers = 84 forward passes.
# Each forward pass captures layers 9, 18, 27 together.
#
# Diagnostic only:
# - does NOT modify `main`
# - does NOT overwrite official result files
# - stores output in `c2_bias_rows`

import numpy as np
import pandas as pd

LAYERS = [9, 18, 27]
DISTANCES = EXP["eviction_distances"]
FILLERS = range(EXP["n_filler_variants"])

PAIRS = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

# --------------------------------------------------
# 1. Verify tokenization before spending GPU compute
# --------------------------------------------------

pair_ids = {}

for needle, distractor in PAIRS.items():

    n_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    d_ids = tok.encode(
        f" {distractor}",
        add_special_tokens=False,
    )

    assert len(n_ids) == 1, (
        f"{needle} is not single-token: {n_ids}"
    )

    assert len(d_ids) == 1, (
        f"{distractor} is not single-token: {d_ids}"
    )

    pair_ids[needle] = (n_ids[0], d_ids[0])


# --------------------------------------------------
# 2. Controlled sweep
# --------------------------------------------------

c2_bias_rows = []

total = len(PAIRS) * len(DISTANCES) * len(list(FILLERS))
done = 0

for stored_needle in PAIRS:

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            spec = ai.build_niah_prompt(
                tok,
                stored_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            # Match the official experiment's validity conditions.
            if not spec["ahn_will_activate"]:
                continue

            if not spec["needle_is_evicted"]:
                continue

            ins = tok(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            # ONE model forward pass captures all three layers.
            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            for L in LAYERS:

                if L not in on.ahn_raw:
                    continue

                o_t = on.o_t(L, pos=-1)

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens,
                    layer=L,
                )

                # Evaluate ALL four pairs from this SAME o_t.
                for tested_needle, distractor in PAIRS.items():

                    needle_id, distractor_id = pair_ids[tested_needle]

                    p_n = ai.token_prob(
                        logits,
                        needle_id,
                    )

                    p_d = ai.token_prob(
                        logits,
                        distractor_id,
                    )

                    # 1e-30 only prevents division by zero.
                    # Unlike the official 1e-12, it does not dominate
                    # the tiny probabilities observed in these rows.
                    ratio = (
                        (p_n + 1e-30) /
                        (p_d + 1e-30)
                    )

                    c2_bias_rows.append({
                        "stored_needle": stored_needle,
                        "tested_needle": tested_needle,
                        "distractor": distractor,
                        "layer": L,
                        "requested_distance": requested_distance,
                        "actual_eviction_distance":
                            spec["actual_eviction_distance"],
                        "filler_idx": filler_idx,
                        "p_needle": p_n,
                        "p_distractor": p_d,
                        "ratio": ratio,
                    })

            done += 1

            if done % 10 == 0 or done == total:
                print(
                    f"{done}/{total} forward passes completed"
                )


# --------------------------------------------------
# 3. Sanity checks
# --------------------------------------------------

df_bias = pd.DataFrame(c2_bias_rows)

print("\n=== SWEEP COMPLETE ===")
print("Forward passes expected:", total)
print("Diagnostic rows:", len(df_bias))

print(
    "Stored needles:",
    sorted(df_bias["stored_needle"].unique())
)

print(
    "Tested needles:",
    sorted(df_bias["tested_needle"].unique())
)

print(
    "Layers:",
    sorted(df_bias["layer"].unique())
)

print(
    "Requested distances:",
    sorted(df_bias["requested_distance"].unique())
)

print(
    "Fillers:",
    sorted(df_bias["filler_idx"].unique())
)

print("\nRows by layer / stored needle / tested needle:")

print(
    df_bias
    .groupby(
        ["layer", "stored_needle", "tested_needle"]
    )
    .size()
    .to_string()
)

In [ ]:
# FULL C2 baseline-corrected analysis
# Uses df_bias already generated.
# No model inference. Does not modify official results.

import numpy as np
import pandas as pd

# Work in log-ratio space:
# log[p(needle)/p(distractor)]
#
# For every exact:
#   layer × distance × filler × tested pair
#
# compare:
#   ratio when THAT needle was stored
# versus
#   mean log-ratio when the other 3 needles were stored.

work = df_bias.copy()

# Numerical guard only.
EPS = 1e-30

work["log_ratio"] = np.log(
    (work["p_needle"] + EPS) /
    (work["p_distractor"] + EPS)
)

corrected = []

group_cols = [
    "layer",
    "requested_distance",
    "filler_idx",
    "tested_needle",
]

for keys, g in work.groupby(group_cols):

    layer, distance, filler, tested = keys

    # The row where the tested needle is actually the stored needle.
    target = g[g["stored_needle"] == tested]

    # Matched baseline: SAME layer/distance/filler/pair,
    # but one of the other three needles was stored.
    baseline = g[g["stored_needle"] != tested]

    assert len(target) == 1, (keys, len(target))
    assert len(baseline) == 3, (keys, len(baseline))

    target_log = float(target["log_ratio"].iloc[0])
    baseline_log = float(baseline["log_ratio"].mean())

    delta_log = target_log - baseline_log

    corrected.append({
        "layer": layer,
        "requested_distance": distance,
        "filler_idx": filler,
        "needle": tested,
        "target_ratio": float(np.exp(target_log)),
        "baseline_ratio": float(np.exp(baseline_log)),
        "corrected_fold": float(np.exp(delta_log)),
        "delta_log_ratio": delta_log,
    })

df_corrected = pd.DataFrame(corrected)

# Expected:
# 3 layers × 7 distances × 3 fillers × 4 needles = 252
assert len(df_corrected) == 252

print("Corrected observations:", len(df_corrected))

print("\n=== FULL BASELINE-CORRECTED C2 ===")

summary = (
    df_corrected
    .groupby(["layer", "needle"])
    .agg(
        n=("corrected_fold", "size"),
        median_corrected_fold=("corrected_fold", "median"),
        geometric_mean_fold=(
            "delta_log_ratio",
            lambda x: float(np.exp(x.mean()))
        ),
        frac_above_1=(
            "corrected_fold",
            lambda x: float((x > 1).mean())
        ),
    )
)

print(summary.to_string())

print("\n=== POOLED BY LAYER ===")

layer_summary = (
    df_corrected
    .groupby("layer")
    .agg(
        n=("corrected_fold", "size"),
        median_corrected_fold=("corrected_fold", "median"),
        geometric_mean_fold=(
            "delta_log_ratio",
            lambda x: float(np.exp(x.mean()))
        ),
        frac_above_1=(
            "corrected_fold",
            lambda x: float((x > 1).mean())
        ),
    )
)

print(layer_summary.to_string())

#### C2 — Full matched baseline-corrected analysis

The pair-specific readout-bias diagnostic was extended across the full
C2 design: 3 layers × 7 eviction distances × 3 filler variants ×
4 needle/distractor pairs (252 matched corrected observations).

For every layer × distance × filler × tested-pair condition, the
needle/distractor log-probability ratio when the tested needle was
actually stored was compared with the same pair's mean log-ratio when
one of the other three needles was stored.

Pooled descriptive results:

| Layer | Median corrected fold | Geometric mean fold | Fraction > 1 |
|---|---:|---:|---:|
| 9  | 1.004 | 1.000 | 52.4% |
| 18 | 1.034 | 1.045 | 61.9% |
| 27 | 1.014 | 0.990 | 54.8% |

These pooled values are descriptive only because the 84 observations
within each layer are not independent.

**Finding:** The large raw differences observed in C2 are strongly
affected by pair-specific readout preferences. After matching each pair
against its own baseline, Layers 9 and 27 remain close to 1×, while
Layer 18 shows a small positive corrected association.

Statistical interpretation is deferred to the condition-level analysis
below, which aggregates the four tested pairs into 21 condition-level
observations per layer.

This result concerns the validity and interpretability of the current
C2 readout statistic. It does not establish that AHN contains no
token-specific information, because such information may not be
recoverable by the current J-Lens/vocabulary readout.

In [ ]:
# FULL C2 prompt-leakage check
# 4 needles × 7 distances × 3 fillers = 84 prompts
# NO model inference / NO GPU / modifies nothing.

WORDS = [
    "Paris", "London",
    "Tokyo", "Osaka",
    "banana", "mango",
    "lantern", "torch",
]

STORED = ["Paris", "Tokyo", "banana", "lantern"]

leaks = []
checked = 0

for stored in STORED:
    for distance in EXP["eviction_distances"]:
        for filler_idx in range(EXP["n_filler_variants"]):

            spec = ai.build_niah_prompt(
                tok,
                stored,
                bundle,
                eviction_distance=distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            prompt_lower = spec["prompt"].lower()
            checked += 1

            for word in WORDS:
                count = prompt_lower.count(word.lower())

                # Intended stored needle should appear exactly once.
                if word == stored:
                    if count != 1:
                        leaks.append({
                            "stored": stored,
                            "distance": distance,
                            "filler": filler_idx,
                            "word": word,
                            "count": count,
                            "problem": "stored needle count != 1",
                        })

                # Every other C2 word should be absent.
                elif count != 0:
                    leaks.append({
                        "stored": stored,
                        "distance": distance,
                        "filler": filler_idx,
                        "word": word,
                        "count": count,
                        "problem": "unexpected word in prompt",
                    })

print("Prompts checked:", checked)
print("Problems found:", len(leaks))

if leaks:
    for x in leaks:
        print(x)
else:
    print("PASS: no C2 target/distractor contamination detected.")

#### C2 — Prompt contamination check

All 84 prompts used in the full C2 diagnostic
(4 needles × 7 distances × 3 filler variants) were checked for
accidental occurrences of all C2 needle and distractor words.

Each prompt contained its intended stored needle exactly once and
contained none of the other tested needles or distractors.

- Prompts checked: 84
- Contamination cases: 0

**Finding:** The observed pair-specific C2 readout preferences cannot
be explained by accidental target/distractor word contamination in the
generated prompts.

This rules out this specific form of prompt-level leakage, but does not
rule out every possible source of experimental bias or leakage.

In [ ]:
# C2 — matched/clustered statistical validation
# ----------------------------------------------
# NO GPU.
# Uses df_corrected only.
#
# Tests:
# 1. Each layer × needle: 21 matched distance×filler conditions.
# 2. Each layer pooled: first average the 4 needles WITHIN each
#    distance×filler condition -> 21 independent condition-level values.
# 3. Bootstrap 95% CI for geometric-mean fold change.
# 4. Two-sided sign-flip permutation test for mean log-fold = 0.
# 5. Holm correction for multiple comparisons.

import numpy as np
import pandas as pd

RNG = np.random.default_rng(42)

N_BOOT = 20_000
N_PERM = 100_000


# --------------------------------------------------
# Sanity checks
# --------------------------------------------------

d = df_corrected.copy()

d["condition"] = list(zip(
    d["requested_distance"],
    d["filler_idx"]
))

assert len(d) == 252
assert d["condition"].nunique() == 21

# Every layer × condition should contain exactly 4 needles.
counts = (
    d.groupby(["layer", "condition"])
     .size()
)

assert (counts == 4).all(), counts[counts != 4]


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def bootstrap_mean_log_ci(x, n_boot=N_BOOT):
    """
    Bootstrap the mean log-fold.
    Returned values are exponentiated, so they are
    geometric-mean fold changes.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)

    idx = RNG.integers(
        0, n,
        size=(n_boot, n)
    )

    boot_means = x[idx].mean(axis=1)

    lo, hi = np.percentile(
        boot_means,
        [2.5, 97.5]
    )

    return (
        float(np.exp(x.mean())),
        float(np.exp(lo)),
        float(np.exp(hi)),
    )


def signflip_pvalue(x, n_perm=N_PERM):
    """
    Two-sided matched sign-flip permutation test.

    H0: mean log-fold = 0.

    The sign of each matched condition is randomly flipped.
    """
    x = np.asarray(x, dtype=float)

    observed = abs(x.mean())

    n = len(x)

    # Generate +/-1 signs.
    signs = RNG.choice(
        np.array([-1.0, 1.0]),
        size=(n_perm, n)
    )

    permuted = (signs * x).mean(axis=1)

    # +1 correction avoids p=0 from finite Monte Carlo sampling.
    p = (
        np.sum(np.abs(permuted) >= observed) + 1
    ) / (n_perm + 1)

    return float(p)


def holm_adjust(pvalues):
    """
    Holm family-wise error correction.
    """
    p = np.asarray(pvalues, dtype=float)

    order = np.argsort(p)
    adjusted = np.empty_like(p)

    running_max = 0.0
    m = len(p)

    for rank, idx in enumerate(order):
        value = (m - rank) * p[idx]
        running_max = max(running_max, value)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


# ==================================================
# A. LAYER × NEEDLE TESTS
# ==================================================

needle_results = []

for (layer, needle), g in d.groupby(
    ["layer", "needle"]
):

    # Exactly one matched observation per
    # distance × filler condition.
    g = g.sort_values(
        ["requested_distance", "filler_idx"]
    )

    x = g["delta_log_ratio"].to_numpy()

    assert len(x) == 21

    fold, lo, hi = bootstrap_mean_log_ci(x)

    p = signflip_pvalue(x)

    needle_results.append({
        "layer": layer,
        "needle": needle,
        "n_conditions": len(x),
        "geometric_mean_fold": fold,
        "ci_low": lo,
        "ci_high": hi,
        "p_raw": p,
    })


needle_stats = pd.DataFrame(needle_results)

# Correct across all 12 layer×needle tests.
needle_stats["p_holm"] = holm_adjust(
    needle_stats["p_raw"].to_numpy()
)

needle_stats["significant_holm_005"] = (
    needle_stats["p_holm"] < 0.05
)


print(
    "=== MATCHED TEST — LAYER × NEEDLE "
    "(Holm corrected across 12 tests) ==="
)

print(
    needle_stats
    .sort_values(["layer", "needle"])
    .to_string(index=False)
)


# ==================================================
# B. POOLED LAYER TESTS
# ==================================================
#
# IMPORTANT:
# Do NOT treat 84 rows as independent.
#
# Within every layer × distance × filler cluster,
# first average the four needle effects.
#
# This gives 21 condition-level observations/layer.

clustered = (
    d.groupby([
        "layer",
        "requested_distance",
        "filler_idx"
    ])["delta_log_ratio"]
    .mean()
    .reset_index(name="cluster_mean_log")
)

layer_results = []

for layer, g in clustered.groupby("layer"):

    g = g.sort_values(
        ["requested_distance", "filler_idx"]
    )

    x = g["cluster_mean_log"].to_numpy()

    assert len(x) == 21

    fold, lo, hi = bootstrap_mean_log_ci(x)

    p = signflip_pvalue(x)

    layer_results.append({
        "layer": layer,
        "n_conditions": len(x),
        "geometric_mean_fold": fold,
        "ci_low": lo,
        "ci_high": hi,
        "p_raw": p,
    })


layer_stats = pd.DataFrame(layer_results)

# Correct across the 3 pooled layer tests.
layer_stats["p_holm"] = holm_adjust(
    layer_stats["p_raw"].to_numpy()
)

layer_stats["significant_holm_005"] = (
    layer_stats["p_holm"] < 0.05
)


print(
    "\n=== MATCHED/CLUSTERED TEST — POOLED BY LAYER "
    "(Holm corrected across 3 tests) ==="
)

print(
    layer_stats
    .sort_values("layer")
    .to_string(index=False)
)

In [ ]:

from scipy import stats

# ============================================================
# CORRECTED CONDITION-LEVEL ANALYSIS
# Unit of analysis = (distance × filler)
# 4 needles are averaged within each condition.
# Expected: 7 distances × 3 fillers = 21 observations/layer
# ============================================================

required = {
    "layer",
    "requested_distance",
    "filler_idx",
    "delta_log_ratio",
}

missing = required - set(df_corrected.columns)
assert not missing, f"Missing columns: {missing}"

# Average the 4 needles inside each condition
cond = (
    df_corrected
    .groupby(
        ["layer", "requested_distance", "filler_idx"],
        as_index=False
    )
    .agg(
        delta_log_ratio=("delta_log_ratio", "mean"),
        n_needles=("delta_log_ratio", "size"),
    )
)

print("=== SANITY CHECK ===")
print("Original rows:", len(df_corrected))
print("Condition rows:", len(cond))
print()
print("Conditions per layer:")
print(cond.groupby("layer").size())
print()
print("Needles per condition:")
print(cond["n_needles"].value_counts().sort_index())

# Every condition should contain all 4 needles
assert (cond["n_needles"] == 4).all(), \
    "ERROR: Some conditions do not contain exactly 4 needles."

# ============================================================
# RESULTS
# ============================================================

results = []

for layer, g in cond.groupby("layer"):

    x = g["delta_log_ratio"].to_numpy()
    n = len(x)

    mean_log = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)

    # 95% t confidence interval
    tcrit = stats.t.ppf(0.975, df=n - 1)

    ci_low_log = mean_log - tcrit * se
    ci_high_log = mean_log + tcrit * se

    # Convert log effects -> fold effects
    fold = np.exp(mean_log)
    ci_low = np.exp(ci_low_log)
    ci_high = np.exp(ci_high_log)

    # Two-sided one-sample t-test against log(effect)=0
    t_stat, p_value = stats.ttest_1samp(x, 0.0)

    results.append({
        "layer": layer,
        "n_conditions": n,
        "mean_log_effect": mean_log,
        "geom_fold": fold,
        "CI_low": ci_low,
        "CI_high": ci_high,
        "t": t_stat,
        "p": p_value,
    })

results = pd.DataFrame(results)

print("\n=== CORRECTED n=21 ANALYSIS ===")
print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)

In [ ]:

# ============================================================
# ROBUSTNESS BATTERY FOR CORRECTED C2 ANALYSIS
#
# 1. Correct n=21 condition-level analysis
# 2. Leave-one-distance-out
# 3. Leave-one-filler-out
# 4. Per-needle analysis
# 5. Fixed blocked permutation test
#
# NO MODEL INFERENCE REQUIRED
# ============================================================

RNG = np.random.default_rng(42)
N_PERM = 20_000


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def find_col(df, candidates, label):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not identify {label} column.\n"
        f"Tried: {candidates}\n"
        f"Available columns:\n{list(df.columns)}"
    )


def summarize_effect(x):
    x = np.asarray(x, dtype=float)
    n = len(x)

    mean_log = np.mean(x)
    se = np.std(x, ddof=1) / np.sqrt(n)

    tcrit = stats.t.ppf(0.975, df=n - 1)

    lo_log = mean_log - tcrit * se
    hi_log = mean_log + tcrit * se

    t_stat, p = stats.ttest_1samp(x, 0.0)

    return {
        "n": n,
        "mean_log": mean_log,
        "fold": np.exp(mean_log),
        "CI_low": np.exp(lo_log),
        "CI_high": np.exp(hi_log),
        "t": t_stat,
        "p": p,
    }


# ------------------------------------------------------------
# Detect needle column
# ------------------------------------------------------------

needle_col = find_col(
    df_corrected,
    ["needle", "stored_needle", "target_needle", "needle_word"],
    "needle"
)

print("Using needle column:", needle_col)


# ============================================================
# 1. CORRECT CONDITION-LEVEL DATA
# ============================================================

cond = (
    df_corrected
    .groupby(
        ["layer", "requested_distance", "filler_idx"],
        as_index=False
    )
    .agg(
        delta_log_ratio=("delta_log_ratio", "mean"),
        n_needles=("delta_log_ratio", "size")
    )
)

assert (cond["n_needles"] == 4).all(), \
    "Some conditions do not contain exactly 4 needles."

print("\n" + "=" * 70)
print("1. CORRECTED CONDITION-LEVEL ANALYSIS")
print("=" * 70)

print("Original rows:", len(df_corrected))
print("Condition rows:", len(cond))
print("\nConditions per layer:")
print(cond.groupby("layer").size())

baseline_rows = []

for layer, g in cond.groupby("layer"):
    r = summarize_effect(g["delta_log_ratio"])
    r["layer"] = layer
    baseline_rows.append(r)

baseline = pd.DataFrame(baseline_rows)[
    ["layer", "n", "mean_log", "fold", "CI_low", "CI_high", "t", "p"]
]

print(
    "\n" +
    baseline.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 2. LEAVE-ONE-DISTANCE-OUT
# ============================================================

print("\n" + "=" * 70)
print("2. LEAVE-ONE-DISTANCE-OUT")
print("=" * 70)

lodo_rows = []

for layer in sorted(cond["layer"].unique()):

    layer_df = cond[
        cond["layer"] == layer
    ]

    for dropped in sorted(
        layer_df["requested_distance"].unique()
    ):

        g = layer_df[
            layer_df["requested_distance"] != dropped
        ]

        r = summarize_effect(
            g["delta_log_ratio"]
        )

        lodo_rows.append({
            "layer": layer,
            "dropped_distance": dropped,
            **r
        })

lodo = pd.DataFrame(lodo_rows)

print(
    lodo[
        [
            "layer",
            "dropped_distance",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 3. LEAVE-ONE-FILLER-OUT
# ============================================================

print("\n" + "=" * 70)
print("3. LEAVE-ONE-FILLER-OUT")
print("=" * 70)

lofo_rows = []

for layer in sorted(cond["layer"].unique()):

    layer_df = cond[
        cond["layer"] == layer
    ]

    for dropped in sorted(
        layer_df["filler_idx"].unique()
    ):

        g = layer_df[
            layer_df["filler_idx"] != dropped
        ]

        r = summarize_effect(
            g["delta_log_ratio"]
        )

        lofo_rows.append({
            "layer": layer,
            "dropped_filler": dropped,
            **r
        })

lofo = pd.DataFrame(lofo_rows)

print(
    lofo[
        [
            "layer",
            "dropped_filler",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 4. PER-NEEDLE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("4. PER-NEEDLE ANALYSIS")
print("=" * 70)

needle_rows = []

for (layer, needle), g in df_corrected.groupby(
    ["layer", needle_col]
):

    r = summarize_effect(
        g["delta_log_ratio"]
    )

    needle_rows.append({
        "layer": layer,
        "needle": needle,
        **r
    })

needle_results = pd.DataFrame(
    needle_rows
)

print(
    needle_results[
        [
            "layer",
            "needle",
            "n",
            "fold",
            "CI_low",
            "CI_high",
            "p"
        ]
    ]
    .sort_values(
        ["layer", "needle"]
    )
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# 5. BLOCKED PERMUTATION TEST
# ============================================================

print("\n" + "=" * 70)
print("5. BLOCKED PERMUTATION TEST")
print("=" * 70)

required_bias = {
    "stored_needle",
    "tested_needle",
    "distractor",
    "layer",
    "requested_distance",
    "filler_idx",
    "p_needle",
    "p_distractor",
}

missing = required_bias - set(
    df_bias.columns
)

assert not missing, \
    f"Missing df_bias columns: {missing}"

bias = df_bias.copy()


# ------------------------------------------------------------
# Check probabilities before logs
# ------------------------------------------------------------

min_prob = min(
    bias["p_needle"].min(),
    bias["p_distractor"].min()
)

print("Minimum probability:", min_prob)

if min_prob <= 0:
    raise ValueError(
        "Zero/negative probability found. "
        "Cannot compute safe log-ratios."
    )


bias["raw_log_ratio"] = (
    np.log(bias["p_needle"])
    -
    np.log(bias["p_distractor"])
)


# ------------------------------------------------------------
# Each block:
# layer × distance × filler × tested pair
#
# Within each block there should be 4 stored prompts.
# ------------------------------------------------------------

block_cols = [
    "layer",
    "requested_distance",
    "filler_idx",
    "tested_needle",
    "distractor"
]

block_sizes = (
    bias
    .groupby(block_cols)
    .size()
)

print("\nBlock-size counts:")
print(
    block_sizes
    .value_counts()
    .sort_index()
)

bad_blocks = block_sizes[
    block_sizes != 4
]

if len(bad_blocks) > 0:
    print("\nBad blocks:")
    print(
        bad_blocks.head(20)
    )
    raise ValueError(
        f"{len(bad_blocks)} blocks do not "
        "contain exactly 4 stored prompts."
    )

print(
    "All matched blocks contain exactly "
    "4 stored prompts: PASS"
)


# ------------------------------------------------------------
# Build blocks
# ------------------------------------------------------------

blocks = []

for key, g in bias.groupby(
    block_cols
):

    # This ordering is important:
    # index 0-3 must correspond to the same
    # stored identities across tested pairs.
    g = g.sort_values(
        "stored_needle"
    )

    vals = (
        g["raw_log_ratio"]
        .to_numpy(dtype=float)
    )

    stored_order = (
        g["stored_needle"]
        .tolist()
    )

    blocks.append({
        "layer": key[0],
        "distance": key[1],
        "filler": key[2],
        "tested_needle": key[3],
        "distractor": key[4],
        "vals": vals,
        "stored_order": stored_order
    })


# ------------------------------------------------------------
# Verify stored ordering is identical everywhere
# ------------------------------------------------------------

all_orders = {
    tuple(b["stored_order"])
    for b in blocks
}

print(
    "\nUnique stored-needle orders:",
    all_orders
)

if len(all_orders) != 1:
    raise ValueError(
        "Stored-needle ordering is not "
        "consistent across blocks."
    )

print(
    "Stored-needle ordering consistent: PASS"
)


# ------------------------------------------------------------
# Observed corrected effect
# ------------------------------------------------------------

observed = (
    cond
    .groupby("layer")[
        "delta_log_ratio"
    ]
    .mean()
    .to_dict()
)

print(
    "\nObserved corrected effects:"
)

for layer, obs in observed.items():
    print(
        f"Layer {layer}: "
        f"mean_log={obs:.6f}, "
        f"fold={np.exp(obs):.6f}x"
    )


# ------------------------------------------------------------
# FIXED BLOCKED PERMUTATION
#
# For each distance × filler condition:
#
# randomly permute the four stored-prompt
# identities exactly once.
#
# The same one-to-one assignment is then
# applied across the four tested pairs.
#
# This avoids sampling targets with replacement.
# ------------------------------------------------------------

perm_results = []

for layer in sorted(
    cond["layer"].unique()
):

    layer_blocks = [
        b
        for b in blocks
        if b["layer"] == layer
    ]

    condition_keys = sorted({
        (
            b["distance"],
            b["filler"]
        )
        for b in layer_blocks
    })

    print(
        f"\nLayer {layer}: "
        f"{len(layer_blocks)} matched blocks, "
        f"{len(condition_keys)} conditions"
    )

    assert len(condition_keys) == 21, (
        f"Expected 21 conditions at "
        f"layer {layer}; "
        f"found {len(condition_keys)}"
    )

    null_stats = np.empty(
        N_PERM,
        dtype=float
    )

    for perm_i in range(
        N_PERM
    ):

        condition_effects = []

        for distance, filler in condition_keys:

            these_blocks = [
                b
                for b in layer_blocks
                if (
                    b["distance"] == distance
                    and
                    b["filler"] == filler
                )
            ]

            # stable order of tested pairs
            these_blocks = sorted(
                these_blocks,
                key=lambda b: (
                    b["tested_needle"],
                    b["distractor"]
                )
            )

            if len(these_blocks) != 4:
                raise ValueError(
                    f"Expected 4 tested-pair blocks "
                    f"for layer={layer}, "
                    f"distance={distance}, "
                    f"filler={filler}; "
                    f"found {len(these_blocks)}"
                )

            # ------------------------------------
            # TRUE ONE-TO-ONE PERMUTATION
            # ------------------------------------

            assignment = (
                RNG.permutation(4)
            )

            pair_effects = []

            for j, b in enumerate(
                these_blocks
            ):

                vals = b["vals"]

                idx = assignment[j]

                pseudo_target = (
                    vals[idx]
                )

                pseudo_baseline = (
                    np.delete(
                        vals,
                        idx
                    ).mean()
                )

                pseudo_delta = (
                    pseudo_target
                    -
                    pseudo_baseline
                )

                pair_effects.append(
                    pseudo_delta
                )

            # Average four tested-pair effects
            # inside this distance × filler condition
            condition_effects.append(
                np.mean(pair_effects)
            )

        # same statistic as corrected n=21 analysis
        null_stats[perm_i] = (
            np.mean(
                condition_effects
            )
        )

    obs = observed[layer]

    p_perm = (
        np.sum(
            np.abs(null_stats)
            >=
            abs(obs)
        )
        + 1
    ) / (
        N_PERM + 1
    )

    perm_results.append({
        "layer": layer,
        "observed_mean_log": obs,
        "observed_fold": np.exp(obs),
        "null_mean": np.mean(null_stats),
        "null_sd": np.std(
            null_stats,
            ddof=1
        ),
        "permutation_p": p_perm
    })


perm_results = pd.DataFrame(
    perm_results
)

print(
    "\n" + "=" * 70
)

print(
    "BLOCKED PERMUTATION RESULTS"
)

print(
    "=" * 70
)

print(
    perm_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}"
    )
)


# ============================================================
# FINAL ROBUSTNESS SUMMARY
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL ROBUSTNESS SUMMARY"
)

print(
    "=" * 70
)


for layer in sorted(
    cond["layer"].unique()
):

    base = baseline[
        baseline["layer"] == layer
    ].iloc[0]

    ld = lodo[
        lodo["layer"] == layer
    ]

    lf = lofo[
        lofo["layer"] == layer
    ]

    nd = needle_results[
        needle_results["layer"] == layer
    ]

    pp = perm_results[
        perm_results["layer"] == layer
    ].iloc[0]

    print(
        f"\nLAYER {layer}"
    )

    print(
        f"  Main corrected fold:    "
        f"{base['fold']:.4f}x"
    )

    print(
        f"  Main 95% CI:            "
        f"[{base['CI_low']:.4f}, "
        f"{base['CI_high']:.4f}]"
    )

    print(
        f"  Main t-test p:          "
        f"{base['p']:.6g}"
    )

    print(
        f"  Leave-distance folds:   "
        f"{ld['fold'].min():.4f}x "
        f"to "
        f"{ld['fold'].max():.4f}x"
    )

    print(
        f"  Leave-filler folds:     "
        f"{lf['fold'].min():.4f}x "
        f"to "
        f"{lf['fold'].max():.4f}x"
    )

    print(
        f"  Per-needle folds:       "
        f"{nd['fold'].min():.4f}x "
        f"to "
        f"{nd['fold'].max():.4f}x"
    )

    print(
        f"  Block permutation p:    "
        f"{pp['permutation_p']:.6g}"
    )


print(
    "\nDONE."
)

print(
    "Send the full FINAL ROBUSTNESS SUMMARY "
    "and BLOCKED PERMUTATION RESULTS for interpretation."
)

In [ ]:


# ============================================================
# C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS
#
# Claude review follow-up #5
#
# Checks:
#   A. actual eviction distance by stored needle
#   B. requested distance balance
#   C. filler balance
#   D. needle position (if available)
#   E. prompt token length (if available)
#   F. local prompt context (if available)
#
# CPU ONLY — NO MODEL INFERENCE
# ============================================================

print("=" * 72)
print("C2 EXPANDED LEAKAGE / PROMPT-CONFOUND CHECKS")
print("=" * 72)


# ------------------------------------------------------------
# 0. Inspect what we actually have
# ------------------------------------------------------------

print("\ndf_bias shape:", df_bias.shape)
print("\ndf_bias columns:")
print(list(df_bias.columns))

required = {
    "stored_needle",
    "layer",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
}

missing = required - set(df_bias.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# One prompt is repeated across tested pairs/layers.
# Reduce to unique prompt-level observations.

prompt_key_candidates = [
    "stored_needle",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
]

for optional in [
    "needle_pos",
    "n_tokens",
    "prompt_tokens",
    "prompt",
]:
    if optional in df_bias.columns:
        prompt_key_candidates.append(optional)

prompts = (
    df_bias[prompt_key_candidates]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nUnique prompt-level rows:", len(prompts))


# ============================================================
# A. REQUESTED DISTANCE BALANCE
# ============================================================

print("\n" + "=" * 72)
print("A. REQUESTED DISTANCE × STORED NEEDLE")
print("=" * 72)

requested_table = pd.crosstab(
    prompts["requested_distance"],
    prompts["stored_needle"]
)

print(requested_table)

balanced_requested = (
    requested_table.nunique(axis=1) == 1
).all()

print(
    "\nBalanced across stored needles:",
    "PASS" if balanced_requested else "CHECK"
)


# ============================================================
# B. FILLER BALANCE
# ============================================================

print("\n" + "=" * 72)
print("B. FILLER × STORED NEEDLE")
print("=" * 72)

filler_table = pd.crosstab(
    prompts["filler_idx"],
    prompts["stored_needle"]
)

print(filler_table)

balanced_filler = (
    filler_table.nunique(axis=1) == 1
).all()

print(
    "\nBalanced across stored needles:",
    "PASS" if balanced_filler else "CHECK"
)


# ============================================================
# C. ACTUAL EVICTION DISTANCE
# ============================================================

print("\n" + "=" * 72)
print("C. ACTUAL EVICTION DISTANCE BY STORED NEEDLE")
print("=" * 72)

eviction_summary = (
    prompts
    .groupby("stored_needle")[
        "actual_eviction_distance"
    ]
    .agg(
        n="count",
        mean="mean",
        std="std",
        min="min",
        median="median",
        max="max"
    )
)

print(eviction_summary)

print("\nMean actual eviction distance by requested distance:")

eviction_by_requested = (
    prompts
    .groupby(
        ["requested_distance", "stored_needle"]
    )["actual_eviction_distance"]
    .mean()
    .unstack()
)

print(eviction_by_requested)

# Range across needles within each requested distance
eviction_spread = (
    eviction_by_requested.max(axis=1)
    -
    eviction_by_requested.min(axis=1)
)

print("\nMax needle-to-needle spread within each requested distance:")
print(eviction_spread)

print(
    "\nLargest spread:",
    eviction_spread.max()
)


# ============================================================
# D. NEEDLE POSITION
# ============================================================

print("\n" + "=" * 72)
print("D. NEEDLE POSITION")
print("=" * 72)

if "needle_pos" in df_bias.columns:

    pos_prompts = (
        df_bias[
            [
                "stored_needle",
                "requested_distance",
                "filler_idx",
                "needle_pos"
            ]
        ]
        .drop_duplicates()
    )

    pos_summary = (
        pos_prompts
        .groupby("stored_needle")[
            "needle_pos"
        ]
        .agg(
            n="count",
            mean="mean",
            std="std",
            min="min",
            median="median",
            max="max"
        )
    )

    print(pos_summary)

    pos_by_condition = (
        pos_prompts
        .pivot_table(
            index=[
                "requested_distance",
                "filler_idx"
            ],
            columns="stored_needle",
            values="needle_pos",
            aggfunc="mean"
        )
    )

    pos_spread = (
        pos_by_condition.max(axis=1)
        -
        pos_by_condition.min(axis=1)
    )

    print(
        "\nLargest within-condition "
        "needle-position spread:",
        pos_spread.max()
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Cannot claim needle-position leakage has been ruled out "
        "from this dataframe."
    )


# ============================================================
# E. PROMPT TOKEN LENGTH
# ============================================================

print("\n" + "=" * 72)
print("E. PROMPT TOKEN LENGTH")
print("=" * 72)

token_length_col = None

for candidate in [
    "n_tokens",
    "prompt_n_tokens",
    "prompt_length",
    "token_length"
]:
    if candidate in df_bias.columns:
        token_length_col = candidate
        break

if token_length_col is not None:

    len_prompts = (
        df_bias[
            [
                "stored_needle",
                "requested_distance",
                "filler_idx",
                token_length_col
            ]
        ]
        .drop_duplicates()
    )

    length_summary = (
        len_prompts
        .groupby("stored_needle")[
            token_length_col
        ]
        .agg(
            n="count",
            mean="mean",
            std="std",
            min="min",
            median="median",
            max="max"
        )
    )

    print(length_summary)

    length_by_condition = (
        len_prompts
        .pivot_table(
            index=[
                "requested_distance",
                "filler_idx"
            ],
            columns="stored_needle",
            values=token_length_col,
            aggfunc="mean"
        )
    )

    length_spread = (
        length_by_condition.max(axis=1)
        -
        length_by_condition.min(axis=1)
    )

    print(
        "\nLargest within-condition "
        "token-length spread:",
        length_spread.max()
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Cannot claim prompt-token-length leakage has been "
        "ruled out from this dataframe."
    )


# ============================================================
# F. LOCAL CONTEXT
# ============================================================

print("\n" + "=" * 72)
print("F. LOCAL CONTEXT AROUND NEEDLE")
print("=" * 72)

context_cols = [
    c for c in [
        "prompt",
        "local_context",
        "needle_context",
        "context"
    ]
    if c in df_bias.columns
]

if context_cols:

    print("Available context columns:", context_cols)

    for c in context_cols:
        print(
            f"\nUnique {c} values:",
            df_bias[c].nunique()
        )

    print(
        "\nContext text is available. "
        "Inspect matched windows around the needle before "
        "declaring this check passed."
    )

else:
    print(
        "NOT AVAILABLE in df_bias.\n"
        "Local filler/context equivalence cannot be verified "
        "from this dataframe."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 72)
print("LEAKAGE CHECK STATUS")
print("=" * 72)

print(
    "Requested-distance balance:",
    "PASS" if balanced_requested else "CHECK"
)

print(
    "Filler balance:",
    "PASS" if balanced_filler else "CHECK"
)

print(
    "Actual eviction distance:",
    "INSPECT TABLE ABOVE"
)

print(
    "Needle position:",
    "AVAILABLE" if "needle_pos" in df_bias.columns
    else "NOT AVAILABLE"
)

print(
    "Prompt token length:",
    "AVAILABLE" if token_length_col is not None
    else "NOT AVAILABLE"
)

print(
    "Local prompt context:",
    "AVAILABLE" if context_cols
    else "NOT AVAILABLE"
)

print("\nDONE.")

In [ ]:
# ============================================================
# C2 TOKEN LENGTH + NEEDLE POSITION CHECK
# CPU ONLY
# ============================================================


print("=" * 72)
print("C2 TOKEN-LENGTH / NEEDLE-POSITION CHECK")
print("=" * 72)

C2_NEEDLES = ["Paris", "Tokyo", "banana", "lantern"]

# Use c2 because it still contains the original metadata
meta = (
    c2[
        c2["needle"].isin(C2_NEEDLES)
    ][
        [
            "needle",
            "requested_distance",
            "filler_idx",
            "n_tokens",
            "needle_pos",
            "eviction_distance",
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("\nUnique metadata rows:", len(meta))

print("\nRows per needle:")
print(meta.groupby("needle").size())


# ============================================================
# 1. TOKEN LENGTH
# ============================================================

print("\n" + "=" * 72)
print("1. TOKEN LENGTH BY NEEDLE")
print("=" * 72)

print(
    meta.groupby("needle")["n_tokens"]
    .agg(["count", "mean", "std", "min", "median", "max"])
)


# ============================================================
# 2. NEEDLE POSITION
# ============================================================

print("\n" + "=" * 72)
print("2. NEEDLE POSITION BY NEEDLE")
print("=" * 72)

print(
    meta.groupby("needle")["needle_pos"]
    .agg(["count", "mean", "std", "min", "median", "max"])
)


# ============================================================
# 3. WITHIN-CONDITION SPREADS
# ============================================================

length_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="n_tokens",
    aggfunc="mean"
)

pos_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="needle_pos",
    aggfunc="mean"
)

evict_pivot = meta.pivot_table(
    index=["requested_distance", "filler_idx"],
    columns="needle",
    values="eviction_distance",
    aggfunc="mean"
)

length_spread = length_pivot.max(axis=1) - length_pivot.min(axis=1)
pos_spread = pos_pivot.max(axis=1) - pos_pivot.min(axis=1)
evict_spread = evict_pivot.max(axis=1) - evict_pivot.min(axis=1)


# ============================================================
# 4. RESULTS
# ============================================================

print("\n" + "=" * 72)
print("WITHIN-CONDITION SPREADS")
print("=" * 72)

print("\nToken-length spread:")
print(length_spread)

print("\nNeedle-position spread:")
print(pos_spread)

print("\nEviction-distance spread:")
print(evict_spread)


# ============================================================
# FINAL
# ============================================================

length_pass = (length_spread == 0).all()
position_pass = (pos_spread == 0).all()
eviction_pass = (evict_spread == 0).all()

print("\n" + "=" * 72)
print("FINAL METADATA CHECK")
print("=" * 72)

print(
    f"Token length:    "
    f"{'PASS' if length_pass else 'CHECK'} "
    f"(max spread={length_spread.max()})"
)

print(
    f"Needle position: "
    f"{'PASS' if position_pass else 'CHECK'} "
    f"(max spread={pos_spread.max()})"
)

print(
    f"Eviction dist.:  "
    f"{'PASS' if eviction_pass else 'CHECK'} "
    f"(max spread={evict_spread.max()})"
)

print("DONE")

### Scrambled-needle/content control — prior supporting experiment

This experiment is retained because it directly tests whether the corrected C2 association survives a content-control manipulation. It is useful supporting evidence about control design, but it is **not a substitute for the mentor-requested single-example WRITE/NOWRITE trace**.


In [ ]:
# ============================================================
# C2 #6 — SCRAMBLED-NEEDLE CONTENT CONTROL
#
# Directly mirrors Cell 41:
#   same build_niah_prompt()
#   same distances
#   same fillers
#   same probe.run()
#   same layers
#   same J-Lens readout
#
# For each C2 needle, replace the stored word with a different
# single-token content word, while still testing the ORIGINAL
# needle/distractor pair.
# ============================================================

LAYERS = [9, 18, 27]
DISTANCES = EXP["eviction_distances"]
FILLERS = range(EXP["n_filler_variants"])

PAIRS = {
    "Paris": "London",
    "Tokyo": "Osaka",
    "banana": "mango",
    "lantern": "torch",
}

# Candidate replacement words.
# We verify them with the SAME tokenization convention as Cell 41:
# tok.encode(f" {word}", add_special_tokens=False)
CONTROL_POOL = [
    "river", "chair", "window", "garden",
    "table", "house", "water", "paper",
    "stone", "music", "green", "cloud",
    "flower", "coffee", "bridge", "forest",
    "silver", "camera", "bottle", "pencil",
    "yellow", "summer", "winter", "kitchen",
    "street", "book", "door", "tree",
]

# ============================================================
# 1. VERIFY ORIGINAL PAIRS + CHOOSE VALID CONTROL WORDS
# ============================================================

pair_ids = {}

for needle, distractor in PAIRS.items():

    n_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    d_ids = tok.encode(
        f" {distractor}",
        add_special_tokens=False,
    )

    assert len(n_ids) == 1, (
        f"{needle} is not single-token: {n_ids}"
    )

    assert len(d_ids) == 1, (
        f"{distractor} is not single-token: {d_ids}"
    )

    pair_ids[needle] = (
        n_ids[0],
        d_ids[0],
    )


# Only retain genuinely single-token controls under the
# exact tokenization convention used by Cell 41.

forbidden_words = (
    set(PAIRS.keys())
    | set(PAIRS.values())
)

valid_controls = []

for word in CONTROL_POOL:

    ids = tok.encode(
        f" {word}",
        add_special_tokens=False,
    )

    if (
        len(ids) == 1
        and word not in forbidden_words
    ):
        valid_controls.append(word)


assert len(valid_controls) >= len(PAIRS), (
    "Not enough verified single-token control words."
)


# Deterministic assignment so the experiment is reproducible.
CONTROL_MAP = {
    needle: valid_controls[i]
    for i, needle in enumerate(PAIRS)
}


print("=" * 72)
print("SCRAMBLED-NEEDLE CONTROL MAP")
print("=" * 72)

for needle, control in CONTROL_MAP.items():

    original_ids = tok.encode(
        f" {needle}",
        add_special_tokens=False,
    )

    control_ids = tok.encode(
        f" {control}",
        add_special_tokens=False,
    )

    print(
        f"{needle:8s} -> {control:8s} | "
        f"{original_ids} -> {control_ids}"
    )

    assert len(original_ids) == 1
    assert len(control_ids) == 1
    assert original_ids[0] != control_ids[0]


# ============================================================
# 2. VERIFY df_bias BEFORE USING IT AS ORIGINAL TARGET
# ============================================================

required_cols = {
    "stored_needle",
    "tested_needle",
    "distractor",
    "layer",
    "requested_distance",
    "actual_eviction_distance",
    "filler_idx",
    "p_needle",
    "p_distractor",
}

missing = required_cols - set(df_bias.columns)

assert not missing, (
    f"df_bias missing columns: {missing}"
)

original_target = df_bias[
    df_bias["stored_needle"]
    == df_bias["tested_needle"]
].copy()

# Expected:
# 4 needles × 7 distances × 3 fillers × 3 layers = 252
assert len(original_target) == 252, (
    f"Expected 252 original target rows, "
    f"found {len(original_target)}"
)

assert (
    original_target.groupby(
        [
            "stored_needle",
            "layer",
            "requested_distance",
            "filler_idx",
        ]
    ).size() == 1
).all()


# ============================================================
# 3. PRE-FLIGHT STRUCTURAL CHECK
#
# Build prompts only first.
# Verify replacing the needle does NOT alter:
#   - prompt token count
#   - needle position
#   - actual eviction distance
#   - AHN activation
#   - eviction status
#
# No probe.run() happens until ALL conditions pass.
# ============================================================

preflight_rows = []

for original_needle, control_needle in CONTROL_MAP.items():

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            original_spec = ai.build_niah_prompt(
                tok,
                original_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            control_spec = ai.build_niah_prompt(
                tok,
                control_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            preflight_rows.append({
                "original_needle": original_needle,
                "control_needle": control_needle,
                "requested_distance": requested_distance,
                "filler_idx": filler_idx,

                "original_n_tokens":
                    original_spec["n_tokens"],

                "control_n_tokens":
                    control_spec["n_tokens"],

                "original_needle_pos":
                    original_spec["needle_pos"],

                "control_needle_pos":
                    control_spec["needle_pos"],

                "original_eviction_distance":
                    original_spec[
                        "actual_eviction_distance"
                    ],

                "control_eviction_distance":
                    control_spec[
                        "actual_eviction_distance"
                    ],

                "original_ahn_active":
                    original_spec["ahn_will_activate"],

                "control_ahn_active":
                    control_spec["ahn_will_activate"],

                "original_evicted":
                    original_spec["needle_is_evicted"],

                "control_evicted":
                    control_spec["needle_is_evicted"],
            })


df_content_preflight = pd.DataFrame(
    preflight_rows
)

# Same 84 conditions as Cell 41.
assert len(df_content_preflight) == 84


df_content_preflight["token_diff"] = (
    df_content_preflight["control_n_tokens"]
    - df_content_preflight["original_n_tokens"]
)

df_content_preflight["position_diff"] = (
    df_content_preflight["control_needle_pos"]
    - df_content_preflight["original_needle_pos"]
)

df_content_preflight["eviction_diff"] = (
    df_content_preflight["control_eviction_distance"]
    - df_content_preflight["original_eviction_distance"]
)


print("\n" + "=" * 72)
print("STRUCTURAL PREFLIGHT")
print("=" * 72)

print(
    "Max |token-count difference|:",
    df_content_preflight[
        "token_diff"
    ].abs().max()
)

print(
    "Max |needle-position difference|:",
    df_content_preflight[
        "position_diff"
    ].abs().max()
)

print(
    "Max |eviction-distance difference|:",
    df_content_preflight[
        "eviction_diff"
    ].abs().max()
)


assert (
    df_content_preflight["token_diff"] == 0
).all(), "STOP: token counts differ."

assert (
    df_content_preflight["position_diff"] == 0
).all(), "STOP: needle positions differ."

assert (
    df_content_preflight["eviction_diff"] == 0
).all(), "STOP: eviction distances differ."

assert (
    df_content_preflight[
        "original_ahn_active"
    ]
    ==
    df_content_preflight[
        "control_ahn_active"
    ]
).all(), "STOP: AHN activation differs."

assert (
    df_content_preflight[
        "original_evicted"
    ]
    ==
    df_content_preflight[
        "control_evicted"
    ]
).all(), "STOP: eviction status differs."

assert (
    df_content_preflight[
        "control_ahn_active"
    ]
).all(), "STOP: a control prompt does not activate AHN."

assert (
    df_content_preflight[
        "control_evicted"
    ]
).all(), "STOP: a control needle is not evicted."


print("Structural preflight: PASS")


# ============================================================
# 4. CONTROL SWEEP
#
# Critical point:
# control_needle is what is STORED,
# but we read out the ORIGINAL needle/distractor pair.
#
# Example:
#   stored = river
#   tested = Paris vs London
#
# This directly tests Claude's content-conditional drift concern.
# ============================================================

c2_content_control_rows = []

total = (
    len(CONTROL_MAP)
    * len(DISTANCES)
    * len(list(FILLERS))
)

done = 0

for original_needle, control_needle in CONTROL_MAP.items():

    needle_id, distractor_id = pair_ids[
        original_needle
    ]

    distractor = PAIRS[
        original_needle
    ]

    for requested_distance in DISTANCES:

        for filler_idx in FILLERS:

            spec = ai.build_niah_prompt(
                tok,
                control_needle,
                bundle,
                eviction_distance=requested_distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            assert spec["ahn_will_activate"]
            assert spec["needle_is_evicted"]

            ins = tok(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            for L in LAYERS:

                if L not in on.ahn_raw:
                    continue

                o_t = on.o_t(
                    L,
                    pos=-1,
                )

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens,
                    layer=L,
                )

                p_n = ai.token_prob(
                    logits,
                    needle_id,
                )

                p_d = ai.token_prob(
                    logits,
                    distractor_id,
                )

                assert p_n > 0
                assert p_d > 0

                c2_content_control_rows.append({
                    "original_needle":
                        original_needle,

                    "control_needle":
                        control_needle,

                    "tested_needle":
                        original_needle,

                    "distractor":
                        distractor,

                    "layer":
                        L,

                    "requested_distance":
                        requested_distance,

                    "actual_eviction_distance":
                        spec[
                            "actual_eviction_distance"
                        ],

                    "filler_idx":
                        filler_idx,

                    "n_tokens":
                        spec["n_tokens"],

                    "needle_pos":
                        spec["needle_pos"],

                    "p_needle":
                        p_n,

                    "p_distractor":
                        p_d,

                    "log_ratio":
                        np.log(p_n)
                        - np.log(p_d),
                })

            done += 1

            if (
                done % 10 == 0
                or done == total
            ):
                print(
                    f"{done}/{total} "
                    "forward passes completed"
                )


df_content_control = pd.DataFrame(
    c2_content_control_rows
)


# ============================================================
# 5. POST-RUN INTEGRITY CHECKS
# ============================================================

assert len(df_content_control) == 252, (
    f"Expected 252 control rows, "
    f"found {len(df_content_control)}"
)

assert (
    df_content_control.groupby(
        [
            "original_needle",
            "layer",
            "requested_distance",
            "filler_idx",
        ]
    ).size() == 1
).all()


# Original target log-ratio.
# Do NOT use EPS here: probabilities were already produced and
# should be strictly positive. Fail loudly if they are not.

assert (
    original_target["p_needle"] > 0
).all()

assert (
    original_target["p_distractor"] > 0
).all()

original_target[
    "original_log_ratio"
] = (
    np.log(
        original_target["p_needle"]
    )
    -
    np.log(
        original_target["p_distractor"]
    )
)


original_for_merge = (
    original_target[
        [
            "stored_needle",
            "layer",
            "requested_distance",
            "filler_idx",
            "actual_eviction_distance",
            "original_log_ratio",
        ]
    ]
    .rename(
        columns={
            "stored_needle":
                "original_needle",

            "actual_eviction_distance":
                "original_eviction_distance",
        }
    )
)


paired_content = original_for_merge.merge(
    df_content_control[
        [
            "original_needle",
            "control_needle",
            "layer",
            "requested_distance",
            "filler_idx",
            "actual_eviction_distance",
            "log_ratio",
        ]
    ].rename(
        columns={
            "actual_eviction_distance":
                "control_eviction_distance",

            "log_ratio":
                "control_log_ratio",
        }
    ),

    on=[
        "original_needle",
        "layer",
        "requested_distance",
        "filler_idx",
    ],

    how="inner",
    validate="one_to_one",
)


assert len(paired_content) == 252


assert (
    paired_content[
        "original_eviction_distance"
    ]
    ==
    paired_content[
        "control_eviction_distance"
    ]
).all()


# ============================================================
# 6. ORIGINAL vs SCRAMBLED-CONTENT EFFECT
#
# Positive:
# original stored word raises its own needle/distractor
# readout relative to the neutral replacement.
#
# Zero:
# original and replacement content behave the same.
#
# Negative:
# original stored word lowers its own pair readout.
# ============================================================

paired_content[
    "delta_log_original_vs_control"
] = (
    paired_content[
        "original_log_ratio"
    ]
    -
    paired_content[
        "control_log_ratio"
    ]
)

paired_content[
    "fold_original_vs_control"
] = np.exp(
    paired_content[
        "delta_log_original_vs_control"
    ]
)


# ============================================================
# 7. CONDITION-LEVEL SUMMARY
#
# Same unit used in corrected analysis:
# average four needle/control comparisons inside each
# layer × distance × filler condition.
# ============================================================

content_cond = (
    paired_content
    .groupby(
        [
            "layer",
            "requested_distance",
            "filler_idx",
        ],
        as_index=False,
    )
    .agg(
        delta_log=(
            "delta_log_original_vs_control",
            "mean",
        ),

        n_pairs=(
            "delta_log_original_vs_control",
            "size",
        ),
    )
)


assert (
    content_cond["n_pairs"] == 4
).all()

assert len(content_cond) == 63


# ============================================================
# 8. REPORT
# ============================================================

print("\n" + "=" * 72)
print("C2 #6 SCRAMBLED-NEEDLE CONTENT CONTROL")
print("=" * 72)

print(
    "\nControl rows:",
    len(df_content_control),
)

print(
    "Paired rows:",
    len(paired_content),
)

print(
    "Condition-level rows:",
    len(content_cond),
)


content_summary_rows = []

for L, g in content_cond.groupby("layer"):

    x = g["delta_log"].to_numpy(
        dtype=float
    )

    n = len(x)

    assert n == 21

    mean_log = x.mean()

    se = (
        x.std(ddof=1)
        / np.sqrt(n)
    )

    tcrit = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_log_low = (
        mean_log
        - tcrit * se
    )

    ci_log_high = (
        mean_log
        + tcrit * se
    )

    t_stat, p_value = (
        stats.ttest_1samp(
            x,
            popmean=0.0,
        )
    )

    content_summary_rows.append({
        "layer":
            L,

        "n_conditions":
            n,

        "mean_log_effect":
            mean_log,

        "geom_fold":
            np.exp(mean_log),

        "CI_low":
            np.exp(ci_log_low),

        "CI_high":
            np.exp(ci_log_high),

        "t":
            t_stat,

        "p":
            p_value,
    })


content_summary = pd.DataFrame(
    content_summary_rows
)


print("\n")
print(
    content_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)


# ============================================================
# 9. PER-NEEDLE RESULT
# ============================================================

print("\n" + "=" * 72)
print("PER-NEEDLE CONTENT CONTROL")
print("=" * 72)

per_needle_content = (
    paired_content
    .groupby(
        [
            "layer",
            "original_needle",
            "control_needle",
        ]
    )
    .agg(
        n=(
            "delta_log_original_vs_control",
            "size",
        ),

        mean_log=(
            "delta_log_original_vs_control",
            "mean",
        ),
    )
    .reset_index()
)

per_needle_content[
    "geom_fold"
] = np.exp(
    per_needle_content[
        "mean_log"
    ]
)

print(
    per_needle_content.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)


print("\nDONE.")

# Mentor-directed C2 end-to-end trace

Gautam's requested trace is:

1. actual input and needle position
2. compression/eviction point
3. AHN `o_t` under **WRITE vs NOWRITE**
4. J-Lens target rank/probability
5. the corresponding C2 content control

The selected diagnostic case is `lantern` vs `garden`, requested distance `64`, filler `2`, with readout pair `lantern/torch`. Prior C2 heterogeneity showed this as a strong negative Layer-18 condition, so it is useful for locating where the expected signal changes direction.

This section is self-contained after the setup cells. **Do not rerun the broad C2/C3 sweeps first.**


In [ ]:
summary = (
    paired_content
    .groupby(["original_needle", "layer"])["delta_log_original_vs_control"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
)

print(summary.to_string(index=False))

### C2 pair-level heterogeneity

The C2 control effect is not consistent across stored needles.

At Layer 18:
- Paris: mean delta = +0.033
- Tokyo: mean delta = +0.002
- banana: mean delta = +0.053
- lantern: mean delta = -0.051

At Layer 27 the disagreement is stronger:
- Paris and Tokyo are positive
- banana and lantern are negative

The within-needle ranges are also large, including both positive and negative conditions.

Therefore, the weak aggregate C2 result is not explained by a single broken summary calculation. The control effect varies substantially by needle and experimental condition, suggesting content/pair dependence or instability in the J-Lens-derived signal.

In [ ]:
worst_lantern = (
    paired_content[
        (paired_content["original_needle"] == "lantern") &
        (paired_content["layer"] == 18)
    ]
    .sort_values("delta_log_original_vs_control")
    .head(5)
)

print(
    worst_lantern[
        [
            "requested_distance",
            "filler_idx",
            "original_log_ratio",
            "control_log_ratio",
            "delta_log_original_vs_control",
            "fold_original_vs_control",
            "original_eviction_distance",
            "control_eviction_distance",
        ]
    ].to_string(index=False)
)

In [ ]:
# Link 1 — input equivalence and actual needle position
# CPU only. No model forward pass.

DEBUG_ORIGINAL   = "lantern"
DEBUG_CONTROL    = "garden"
DEBUG_DISTRACTOR = "torch"
DEBUG_DIST       = 64
DEBUG_FILLER     = 2

orig_ids = tok.encode(f" {DEBUG_ORIGINAL}", add_special_tokens=False)
ctrl_ids = tok.encode(f" {DEBUG_CONTROL}", add_special_tokens=False)
dist_ids = tok.encode(f" {DEBUG_DISTRACTOR}", add_special_tokens=False)

assert len(orig_ids) == 1, (DEBUG_ORIGINAL, orig_ids)
assert len(ctrl_ids) == 1, (DEBUG_CONTROL, ctrl_ids)
assert len(dist_ids) == 1, (DEBUG_DISTRACTOR, dist_ids)

ORIG_ID, CTRL_ID, DIST_ID = orig_ids[0], ctrl_ids[0], dist_ids[0]

spec_real = ai.build_niah_prompt(
    tok, DEBUG_ORIGINAL, bundle,
    eviction_distance=DEBUG_DIST,
    in_window=False,
    filler_idx=DEBUG_FILLER,
)
spec_ctrl = ai.build_niah_prompt(
    tok, DEBUG_CONTROL, bundle,
    eviction_distance=DEBUG_DIST,
    in_window=False,
    filler_idx=DEBUG_FILLER,
)

ins_real = tok(spec_real["prompt"], return_tensors="pt")
ins_ctrl = tok(spec_ctrl["prompt"], return_tensors="pt")

ids_real = ins_real["input_ids"][0].tolist()
ids_ctrl = ins_ctrl["input_ids"][0].tolist()

assert len(ids_real) == len(ids_ctrl), "real/control token lengths differ"
assert spec_real["needle_pos"] == spec_ctrl["needle_pos"], "needle positions differ"

diff_positions = [
    i for i, (a, b) in enumerate(zip(ids_real, ids_ctrl))
    if a != b
]

print(
    f"case: {DEBUG_ORIGINAL!r} vs {DEBUG_CONTROL!r}; "
    f"readout pair {DEBUG_ORIGINAL}/{DEBUG_DISTRACTOR}; "
    f"requested distance={DEBUG_DIST}; filler={DEBUG_FILLER}"
)
print(f"tokenized length: {len(ids_real)}")
print(
    "actual needle position:",
    spec_real["needle_pos"],
    "(real/control matched)"
)
print("differing positions:", diff_positions)

if len(diff_positions) == 1:
    p = diff_positions[0]
    print(
        f"only input difference @ {p}: "
        f"{tok.decode([ids_real[p]])!r} -> {tok.decode([ids_ctrl[p]])!r}"
    )
else:
    print("WARNING: expected exactly one differing token")

pos = spec_real["needle_pos"]
lo = max(0, pos - 6)
hi = min(len(ids_real), pos + 7)

print("\nreal/control tokens around needle position:")
for i in range(lo, hi):
    marker = "  <-- needle_pos" if i == pos else ""
    print(
        f"[{i}] real={tok.decode([ids_real[i]])!r:14s} "
        f"ctrl={tok.decode([ids_ctrl[i]])!r}{marker}"
    )


In [ ]:
print("compression_boundary:",
      spec_real["compression_boundary"],
      spec_ctrl["compression_boundary"])

print("actual_eviction_distance:",
      spec_real["actual_eviction_distance"],
      spec_ctrl["actual_eviction_distance"])

print("needle_is_evicted:",
      spec_real["needle_is_evicted"],
      spec_ctrl["needle_is_evicted"])

In [ ]:
# Link 3 setup — run only the three forward passes needed for this trace.
# capture_residual=False keeps memory use lower; this trace only needs AHN o_t.

device = next(bundle.model.parameters()).device
ins_real = {k: v.to(device) for k, v in ins_real.items()}
ins_ctrl = {k: v.to(device) for k, v in ins_ctrl.items()}

# Target prompt with AHN writes enabled.
on_real = probe.run(
    ins_real,
    nowrite=False,
    layers=EXP["layers"],
    capture_residual=False,
)

# Same target prompt with AHN writes disabled.
off_real = probe.run(
    ins_real,
    nowrite=True,
    layers=EXP["layers"],
    capture_residual=False,
)

# Corresponding C2 content-control prompt with writes enabled.
on_ctrl = probe.run(
    ins_ctrl,
    nowrite=False,
    layers=EXP["layers"],
    capture_residual=False,
)

print("completed: target WRITE, target NOWRITE, control WRITE")


In [ ]:
# Link 3 — log actual AHN o_t for WRITE/NOWRITE and the corresponding control.
# Print compact vector slices rather than all 2048 values.

import torch.nn.functional as F

for L in EXP["layers"]:
    o_write = on_real.o_t(L, pos=-1).float()
    o_nowrite = off_real.o_t(L, pos=-1).float()
    o_ctrl = on_ctrl.o_t(L, pos=-1).float()

    write_nowrite_diff = o_write - o_nowrite
    write_ctrl_diff = o_write - o_ctrl

    write_norm = o_write.norm().item()
    nowrite_norm = o_nowrite.norm().item()
    ctrl_norm = o_ctrl.norm().item()

    print(f"\nLayer {L}")
    print("WRITE   ||o_t||:", write_norm)
    print("NOWRITE ||o_t||:", nowrite_norm)
    print("CONTROL ||o_t||:", ctrl_norm)

    print(
        "WRITE-NOWRITE ||diff||:",
        write_nowrite_diff.norm().item()
    )
    print(
        "WRITE-CONTROL ||diff||:",
        write_ctrl_diff.norm().item()
    )
    print(
        "WRITE-CONTROL relative_diff:",
        write_ctrl_diff.norm().item() / max(write_norm, 1e-12)
    )
    print(
        "WRITE-CONTROL cosine:",
        F.cosine_similarity(
            o_write.flatten(),
            o_ctrl.flatten(),
            dim=0,
        ).item()
    )

    print("WRITE   o_t[:8]:", o_write.flatten()[:8].tolist())
    print("NOWRITE o_t[:8]:", o_nowrite.flatten()[:8].tolist())
    print("CONTROL o_t[:8]:", o_ctrl.flatten()[:8].tolist())


In [ ]:
# Link 4 + Link 5 — J-Lens target rank/probability and matching C2 control.
# J-Lens is applied to target WRITE and control WRITE.
# NOWRITE is logged at the o_t level above; its AHN output is expected to be zero
# by construction, so we do not interpret a vocabulary ranking from that zero vector.

import pandas as pd

needle_id = tok.encode(
    f" {DEBUG_ORIGINAL}",
    add_special_tokens=False,
)[0]
distractor_id = tok.encode(
    f" {DEBUG_DISTRACTOR}",
    add_special_tokens=False,
)[0]

trace_rows = []

for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1)
    o_ctrl = on_ctrl.o_t(L, pos=-1)

    lg_real = ai.readout_logits(
        o_real,
        bundle,
        lens=lens if EXP["use_jlens"] else None,
        layer=L,
    )
    lg_ctrl = ai.readout_logits(
        o_ctrl,
        bundle,
        lens=lens if EXP["use_jlens"] else None,
        layer=L,
    )

    logp_real = torch.log_softmax(lg_real.float(), dim=-1)
    logp_ctrl = torch.log_softmax(lg_ctrl.float(), dim=-1)

    real_log_ratio = (
        logp_real[needle_id] - logp_real[distractor_id]
    ).item()
    ctrl_log_ratio = (
        logp_ctrl[needle_id] - logp_ctrl[distractor_id]
    ).item()

    row = {
        "layer": L,
        "target_rank_WRITE": ai.token_rank(lg_real, needle_id),
        "target_prob_WRITE": ai.token_prob(lg_real, needle_id),
        "target_rank_CONTROL": ai.token_rank(lg_ctrl, needle_id),
        "target_prob_CONTROL": ai.token_prob(lg_ctrl, needle_id),
        "distractor_prob_WRITE": ai.token_prob(lg_real, distractor_id),
        "distractor_prob_CONTROL": ai.token_prob(lg_ctrl, distractor_id),
        "log_ratio_WRITE": real_log_ratio,
        "log_ratio_CONTROL": ctrl_log_ratio,
        "delta_log_ratio": real_log_ratio - ctrl_log_ratio,
    }
    trace_rows.append(row)

trace_df = pd.DataFrame(trace_rows)

print(
    f"readout={READOUT}; target={DEBUG_ORIGINAL} ({needle_id}); "
    f"distractor={DEBUG_DISTRACTOR} ({distractor_id})"
)
print(trace_df.to_string(index=False))


In [ ]:
# Optional bookkeeping cross-check against the previously computed C2 control table.
# The mentor trace itself does not depend on `paired_content`.

if "paired_content" not in globals():
    print(
        "paired_content is not loaded. "
        "The live target/control trace above is complete; skip this cross-check."
    )
else:
    saved = paired_content[
        (paired_content["original_needle"] == DEBUG_ORIGINAL) &
        (paired_content["requested_distance"] == DEBUG_DIST) &
        (paired_content["filler_idx"] == DEBUG_FILLER)
    ][[
        "layer",
        "original_log_ratio",
        "control_log_ratio",
        "delta_log_original_vs_control",
        "original_eviction_distance",
        "control_eviction_distance",
    ]].copy()

    check_trace = trace_df.merge(saved, on="layer", validate="one_to_one")
    check_trace["live_minus_saved_delta"] = (
        check_trace["delta_log_ratio"] -
        check_trace["delta_log_original_vs_control"]
    )

    print(check_trace.to_string(index=False))
    print(
        "\nmax |live - saved delta|:",
        check_trace["live_minus_saved_delta"].abs().max()
    )


### Mentor trace — interpretation gate

Use the **new live outputs above** to mark each link. Do not reuse the old numeric conclusion because the previous trace did not explicitly log target WRITE vs NOWRITE `o_t`.

Required checks:

- **Input:** actual needle position is correct; real/control prompts are structurally matched.
- **Eviction:** compression boundary and actual eviction distance match; both stored words are evicted.
- **AHN `o_t`:** record target WRITE and NOWRITE norms/vector slices, plus target-vs-control difference.
- **J-Lens:** record target rank/probability for target WRITE and content-control WRITE.
- **Control:** record the live target-vs-distractor delta and, when available, compare it with the previously saved C2 row.

Only after these live values are checked should a new PASS / FAIL / NEEDS CHECKING conclusion be written.


In [ ]:
needle_id = tokenizer.encode(
    f" {DEBUG_ORIGINAL}",
    add_special_tokens=False
)[0]

distractor_id = tokenizer.encode(
    f" {DEBUG_DISTRACTOR}",
    add_special_tokens=False
)[0]

print("pair:", DEBUG_ORIGINAL, "/", DEBUG_DISTRACTOR)

for L in EXP["layers"]:
    o_real = on_real.o_t(L, pos=-1)
    o_ctrl = on_ctrl.o_t(L, pos=-1)

    # J-Lens
    j_real = ai.readout_logits(
        o_real,
        bundle,
        lens=lens,
        layer=L
    )

    j_ctrl = ai.readout_logits(
        o_ctrl,
        bundle,
        lens=lens,
        layer=L
    )

    # Plain readout
    plain_real = ai.readout_logits(
        o_real,
        bundle,
        lens=None
    )

    plain_ctrl = ai.readout_logits(
        o_ctrl,
        bundle,
        lens=None
    )

    j_delta = (
        (j_real[needle_id] - j_real[distractor_id])
        -
        (j_ctrl[needle_id] - j_ctrl[distractor_id])
    )

    plain_delta = (
        (plain_real[needle_id] - plain_real[distractor_id])
        -
        (plain_ctrl[needle_id] - plain_ctrl[distractor_id])
    )

    print(f"\nLayer {L}")
    print("J-Lens delta:", j_delta.item())
    print("Plain delta: ", plain_delta.item())

    print("J real rank needle:",
          ai.token_rank(j_real, needle_id))
    print("J ctrl rank needle:",
          ai.token_rank(j_ctrl, needle_id))

    print("Plain real rank needle:",
          ai.token_rank(plain_real, needle_id))
    print("Plain ctrl rank needle:",
          ai.token_rank(plain_ctrl, needle_id))

### Plain-readout vs J-Lens diagnostic

For the strongest negative C2 case (`lantern` vs `garden`,
readout pair `lantern/torch`), the negative corrected effect is present
under both readout methods.

Layer 18:
- J-Lens delta = -1.0492
- Plain readout delta = -0.6560

Therefore, the negative C2 direction is not created solely by the J-Lens
transformation. The same direction is already present when the AHN `o_t`
state is decoded with the plain vocabulary readout.

This shifts the likely explanation away from a J-Lens-specific sign reversal
and toward either content-dependent AHN state geometry or the target-vs-
distractor control metric itself.

In [ ]:
L = 18

o_real = on_real.o_t(L, pos=-1)
o_ctrl = on_ctrl.o_t(L, pos=-1)

lg_real = ai.readout_logits(
    o_real,
    bundle,
    lens=lens,
    layer=L
)

lg_ctrl = ai.readout_logits(
    o_ctrl,
    bundle,
    lens=lens,
    layer=L
)

lantern_real = lg_real[needle_id].item()
lantern_ctrl = lg_ctrl[needle_id].item()

torch_real = lg_real[distractor_id].item()
torch_ctrl = lg_ctrl[distractor_id].item()

print("Layer 18")
print("lantern real logit:", lantern_real)
print("lantern ctrl logit:", lantern_ctrl)
print("lantern change real-ctrl:", lantern_real - lantern_ctrl)

print()
print("torch real logit:", torch_real)
print("torch ctrl logit:", torch_ctrl)
print("torch change real-ctrl:", torch_real - torch_ctrl)

print()
print("pair delta:",
      (lantern_real - torch_real) -
      (lantern_ctrl - torch_ctrl))

### Why the Layer-18 C2 delta is negative

For the `lantern` vs `garden` condition:

- `lantern` logit increases by +0.687
- `torch` logit increases by +1.736

Therefore the stored `lantern` does strengthen the target token, but it
strengthens the semantically related distractor `torch` even more.

This produces the negative corrected pair delta:

`(+0.687) - (+1.736) = -1.049`

So this C2 failure is not simply "the model does not retain lantern."
Instead, the target-vs-distractor metric is strongly affected by how the
stored content changes both members of the semantic pair.

## Supporting C2-v2 multi-control redesign — prior result

These cells are retained because they test an alternative control design after the original target-vs-distractor C2 metric showed strong pair dependence. They are supporting evidence, not a replacement for the current mentor trace.


In [ ]:
CONTROL_CANDIDATES = [
    "river",
    "chair",
    "window",
    "garden",
    "table",
    "house",
    "water",
    "paper",
    "stone",
    "music",
    "green",
    "cloud",
]

targets = ["Paris", "Tokyo", "banana", "lantern"]

valid_controls = []

for word in CONTROL_CANDIDATES:
    ids = tokenizer.encode(
        f" {word}",
        add_special_tokens=False
    )

    # must be exactly one token
    if len(ids) != 1:
        continue

    # don't allow target words themselves
    if word in targets:
        continue

    valid_controls.append((word, ids[0]))

print("Valid single-token controls:")
for word, tid in valid_controls:
    print(f"{word:10s} -> {tid}")

In [ ]:
PILOT_CONTROLS = ["river", "chair", "window", "garden"]
TARGETS = ["Paris", "Tokyo", "banana", "lantern"]

problems = []

for target in TARGETS:
    for control in PILOT_CONTROLS:
        for distance in EXP["eviction_distances"]:
            for filler_idx in range(EXP["n_filler_variants"]):

                spec = ai.build_niah_prompt(
                    tokenizer,
                    control,
                    bundle,
                    eviction_distance=distance,
                    in_window=False,
                    filler_idx=filler_idx,
                )

                prompt_lower = spec["prompt"].lower()

                # target should not accidentally already appear
                if target.lower() in prompt_lower:
                    problems.append(
                        (target, control, distance, filler_idx, "target leaked")
                    )

                # control should appear exactly once as the stored word
                if prompt_lower.count(control.lower()) != 1:
                    problems.append(
                        (
                            target,
                            control,
                            distance,
                            filler_idx,
                            f"control count={prompt_lower.count(control.lower())}",
                        )
                    )

print("Problems found:", len(problems))

for row in problems[:20]:
    print(row)

if not problems:
    print("PASS: pilot control prompts are structurally clean.")

In [ ]:
# ============================================================
# C2-v2 PILOT — multi-control target-only baseline
# 84 forward passes total
# ============================================================

import pandas as pd

PILOT_CONTROLS = ["river", "chair", "window", "garden"]
TARGETS = ["Paris", "Tokyo", "banana", "lantern"]
LAYERS = [9, 18, 27]

target_ids = {
    target: tokenizer.encode(
        f" {target}",
        add_special_tokens=False
    )[0]
    for target in TARGETS
}

pilot_rows = []

total = (
    len(PILOT_CONTROLS)
    * len(EXP["eviction_distances"])
    * EXP["n_filler_variants"]
)

done = 0

for control in PILOT_CONTROLS:

    for distance in EXP["eviction_distances"]:

        for filler_idx in range(EXP["n_filler_variants"]):

            spec = ai.build_niah_prompt(
                tokenizer,
                control,
                bundle,
                eviction_distance=distance,
                in_window=False,
                filler_idx=filler_idx,
            )

            assert spec["needle_is_evicted"], (
                control,
                distance,
                filler_idx,
                spec["actual_eviction_distance"],
            )

            ins = tokenizer(
                spec["prompt"],
                return_tensors="pt",
            ).to(bundle.model.device)

            on = probe.run(
                ins,
                nowrite=False,
                layers=LAYERS,
                capture_residual=False,
            )

            # One AHN state -> score ALL four targets
            for L in LAYERS:

                o_t = on.o_t(L, pos=-1)

                logits = ai.readout_logits(
                    o_t,
                    bundle,
                    lens=lens if EXP["use_jlens"] else None,
                    layer=L,
                )

                for target in TARGETS:

                    p_target = ai.token_prob(
                        logits,
                        target_ids[target],
                    )

                    pilot_rows.append({
                        "control_needle": control,
                        "tested_target": target,
                        "layer": L,
                        "requested_distance": distance,
                        "actual_eviction_distance":
                            spec["actual_eviction_distance"],
                        "filler_idx": filler_idx,
                        "p_target": p_target,
                    })

            done += 1

            print(
                f"\r{done}/{total} forward passes",
                end="",
                flush=True,
            )

print("\nDone.")

df_c2_multicontrol = pd.DataFrame(pilot_rows)

print("rows:", len(df_c2_multicontrol))
print(df_c2_multicontrol.head())

In [ ]:
# ------------------------------------------------------------
# Original condition: target itself was stored
# ------------------------------------------------------------
orig = df_bias[
    df_bias["stored_needle"] == df_bias["tested_needle"]
][[
    "stored_needle",
    "layer",
    "requested_distance",
    "filler_idx",
    "p_needle",
]].copy()

orig = orig.rename(columns={
    "stored_needle": "target",
    "p_needle": "p_target_original",
})

# ------------------------------------------------------------
# Multi-control baseline:
# mean log p(target) across the 4 unrelated controls
# ------------------------------------------------------------
ctrl = df_c2_multicontrol.copy()

ctrl["log_p_target"] = np.log(ctrl["p_target"] + 1e-30)

ctrl_mean = (
    ctrl
    .groupby([
        "tested_target",
        "layer",
        "requested_distance",
        "filler_idx",
    ])
    .agg(
        mean_log_p_control=("log_p_target", "mean"),
        std_log_p_control=("log_p_target", "std"),
        n_controls=("control_needle", "nunique"),
    )
    .reset_index()
    .rename(columns={"tested_target": "target"})
)

# ------------------------------------------------------------
# Match original condition to multi-control baseline
# ------------------------------------------------------------
c2_v2 = orig.merge(
    ctrl_mean,
    on=[
        "target",
        "layer",
        "requested_distance",
        "filler_idx",
    ],
    validate="one_to_one",
)

c2_v2["delta_log_target_multicontrol"] = (
    np.log(c2_v2["p_target_original"] + 1e-30)
    - c2_v2["mean_log_p_control"]
)

# ------------------------------------------------------------
# Summary by target/layer
# ------------------------------------------------------------
summary_c2_v2 = (
    c2_v2
    .groupby(["target", "layer"])["delta_log_target_multicontrol"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
)

print(summary_c2_v2.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# ============================================================
# FINAL C2-v2 CONDITION-LEVEL ANALYSIS
# ============================================================

# 1) Average the 4 targets within each distance × filler condition
condition_level = (
    c2_v2
    .groupby([
        "layer",
        "requested_distance",
        "filler_idx",
    ])["delta_log_target_multicontrol"]
    .mean()
    .reset_index(name="condition_delta")
)

print("condition counts per layer:")
print(condition_level.groupby("layer").size())
print()

# 2) Summary + 95% CI + one-sample t-test against 0
final_rows = []

for L in sorted(condition_level["layer"].unique()):

    x = condition_level.loc[
        condition_level["layer"] == L,
        "condition_delta"
    ].to_numpy()

    n = len(x)
    mean_log = x.mean()
    sd = x.std(ddof=1)
    se = sd / np.sqrt(n)

    tcrit = stats.t.ppf(0.975, df=n - 1)

    ci_low_log = mean_log - tcrit * se
    ci_high_log = mean_log + tcrit * se

    t_stat, p_value = stats.ttest_1samp(x, popmean=0.0)

    final_rows.append({
        "layer": L,
        "n_conditions": n,
        "mean_log_effect": mean_log,
        "fold_change": np.exp(mean_log),
        "ci_low_fold": np.exp(ci_low_log),
        "ci_high_fold": np.exp(ci_high_log),
        "t_stat": t_stat,
        "p_value": p_value,
    })

final_c2_v2 = pd.DataFrame(final_rows)

print(final_c2_v2.to_string(
    index=False,
    formatters={
        "mean_log_effect": "{:.6f}".format,
        "fold_change": "{:.4f}".format,
        "ci_low_fold": "{:.4f}".format,
        "ci_high_fold": "{:.4f}".format,
        "t_stat": "{:.4f}".format,
        "p_value": "{:.6g}".format,
    }
))

### Prior C2-v2 condition-level result

The multi-control target-only redesign was previously evaluated at the condition level (21 distance × filler observations per layer):

- Layer 9: 0.909×, 95% CI [0.843, 0.980], p = 0.015
- Layer 18: 1.045×, 95% CI [0.902, 1.212], p = 0.538
- Layer 27: 0.860×, 95% CI [0.760, 0.974], p = 0.020

This is useful supporting evidence about C2 control design, but it **does not close the current debugging task**. The immediate next step remains the mentor-directed single-example trace above.

All interpretations remain conditional on the J-Lens validation status.
